# 금정산 2021 정사영상·Meta CHM → 국내 공식 지상부 탄소

**Google Colab / T4 GPU용 새 노트북입니다. Windows에서 과학 패키지를 실행하지 않습니다.**
서정범 야장50개에서 사진·높이 추정과 국내 상대생장식 실측 기반 값을 비교합니다.
기본 참조는DBH≥10cm, 보조는≥6cm입니다. **줄기+가지+잎 지상부 탄소만 계산하며 뿌리는 제외합니다.**
개별 나무 위치가 없어F1/precision/recall·헝가리안 개체매칭을 수행하지 않습니다.
NEON의고정 탐지·CHM최대·상자CD 함수만재사용하며 한국수종공식으로교체합니다.

실행 순서: 아래 업로드 안내 → 패키지 설치 → 런타임 재시작이 필요하면 재시작 → Drive마운트부터 위에서 아래로 실행.
첫 실행은50개crop·COG접속·GPU·Drive속도에 따라 수십분 이상 걸릴 수 있습니다.
원본9장약3GB는수정하지않으며 crop과탐지를Drive에캐시합니다. 공개저장소로자료를전송하는코드는없습니다.

## 00. Drive 업로드 구조

이노트북하나만Colab에올리면됩니다.아래입력자료는회사승인범위내개인/조직Drive에직접업로드하세요.
Drive루트기본값은 `/content/drive/MyDrive/geumjeongsan`입니다.

```text
geumjeongsan/
  gedi_geumjeongsan/data/
    ortho/2021/       # 지정9개 (B060)…tif + 대응XML9개, 이름유지
    field/           # seo2011_plots.csv, seo2011_trees.csv, seo2011_layers.csv
    reference/       # README.md, nifos2014_allometry.csv,
                     # nifos2014_carbon_factors.csv, nifos2010_carbon_fraction.csv
    forest_map/
      2025_임상도/{금정구,북구,양산시}/  # .shp/.shx/.dbf/.prj/.cpg 등 함께
      등산로/{금정구,북구,양산시}/       # 동일
    raw/meta_chm/    # 이미받은native CHM quadkey TIFF가있으면선택업로드
```

자료를 `geumjeongsan/data/`에바로배치한경우도자동인식합니다. PDF원문이나임목축적KOSIS파일은필요없습니다.
NEON커밋·25cm기준선·재사용함수는노트북에내장되어추가모듈업로드가필요없습니다.
Meta CHM은이미받은quad파일을먼저찾고없으면해당quad의필요COG window만원격으로읽습니다.
HF모델과Jucker계수원문은공개서버에서받습니다.국내정사영상을공개서버로전송하지않습니다.

## 01. Colab 패키지 설치

설치후재시작안내가나오면재시작하고02번부터실행하세요. NEON과같은DeepForest2.1.0을사용합니다.

In [ ]:
%pip -q install "deepforest==2.1.0" "scikit-learn==1.6.1" rasterio geopandas shapely pyproj numpy scipy scikit-image matplotlib requests tabulate

## 02. Drive 마운트·설정

회사승인입력만사용하세요.이노트북의출력과캐시는아래ROOT하위에저장됩니다. HF캐시는Drive와분리합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os
ROOT=Path('/content/drive/MyDrive/geumjeongsan')
ROOT.mkdir(parents=True,exist_ok=True)
EXTRA_CHM_DIRS=[
    '/content/drive/MyDrive/gedi_geumjeongsan/data/raw/meta_chm',
    '/content/drive/MyDrive/gedi_hongcheon/data/raw/chm',
]
os.environ['HF_HOME']='/content/hf_cache'
os.environ['HF_HUB_CACHE']='/content/hf_cache/hub'
os.environ['TORCH_HOME']='/content/torch_cache'
Path('/content/hf_cache').mkdir(parents=True,exist_ok=True)
HF_MODEL_PROVENANCE={'model_name': 'weecology/deepforest-tree', 'source_revision': 'main', 'resolved_revision': 'cc21436bc5d572dde8ff5f93c1e71a32f563cace', 'deepforest_version': '2.1.0', 'limitation': '3차 결과는 main으로만 기록되어 당시 가중치 SHA 동일 여부는 직접 검증할 수 없음'}
NEON_25CM_BASELINE={'estimate_reference_ratio': 0.5279927701802647, 'F1_visible': 0.44061302681992337, 'carbon_RMSE_tC_ha': 94.17650439651679, 'reference_mean': 134.50074380222082, 'estimate_mean': 71.01542031144064, 'r': 0.44349260250950795, 'MAE': 63.612788219199835}
GEUM_CODE_SHA='9a1611f033331e947235f7e02157299f9e69bf7ec9567d9cbe70e6f1705e6315'

## 03. 순수계산 함수

도엽·quadkey·국내수종대응·부위별공식입니다.실측과영상추정이같은국내공식을사용하며평균대용은예측질량을평균합니다.

In [ ]:
"""Pure Python helpers shared by Colab and standard-library-only local checks."""
import csv
import hashlib
import math
import re
import struct
from pathlib import Path

SHEETS = ('35909073','35909082','35909083','35909091','35909092','35909093','35913001','35913003','35913013')
CONTROL_5179 = (1140963,1693231,1143384,1696139)

def sheet_bounds(sheet):
    if not re.fullmatch(r'\d{8}', str(sheet)):
        raise ValueError('8자리 1:5000 도엽 번호 필요')
    sheet = str(sheet); latitude=int(sheet[:2]); longitude=120+int(sheet[2])
    big=int(sheet[3:5])-1; small=int(sheet[5:])-1
    if not (0<=big<16 and 0<=small<100):raise ValueError('잘못된 도엽 번호')
    row,column=divmod(big,4); sr,sc=divmod(small,10)
    west=longitude+column*.25+sc*.025
    north=latitude+1-row*.25-sr*.025
    return west,north-.025,west+.025,north

def quadkey_xy(x,y,z=9):
    return ''.join(str((1 if x&(1<<i) else 0)+(2 if y&(1<<i) else 0)) for i in range(z-1,-1,-1))

def quadkey_bounds(quad):
    x=y=0;z=len(quad)
    for c in quad:
        digit=int(c)
        if digit not in (0,1,2,3):raise ValueError('invalid quadkey')
        x=x*2+(digit&1);y=y*2+(digit>>1)
    def latitude(v):return math.degrees(math.atan(math.sinh(math.pi*(1-2*v/(2**z)))))
    return x/(2**z)*360-180,latitude(y+1),(x+1)/(2**z)*360-180,latitude(y)

def quadkeys_for_bounds(bounds,z=9):
    west,south,east,north=bounds
    def xy(lon,lat):
        lat=max(-85.05112878,min(85.05112878,lat)); n=2**z
        return max(0,min(n-1,int((lon+180)/360*n))),max(0,min(n-1,int((1-math.asinh(math.tan(math.radians(lat)))/math.pi)/2*n)))
    x0,y0=xy(west,north); x1,y1=xy(east,south)
    return [quadkey_xy(x,y,z) for y in range(y0,y1+1) for x in range(x0,x1+1)]

def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def read_csv_rows(path):
    with Path(path).open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))

def tiff_header(path):
    """Read tiny classic-TIFF IFD metadata only; never read pixel arrays."""
    with Path(path).open('rb') as f:
        order={'II':'<','MM':'>'}[f.read(2).decode()]
        if struct.unpack(order+'H',f.read(2))[0]!=42:raise ValueError('Classic TIFF only')
        f.seek(struct.unpack(order+'I',f.read(4))[0]); count=struct.unpack(order+'H',f.read(2))[0]
        tags={}
        for _ in range(count):
            tag,kind,n,raw=struct.unpack(order+'HHI4s',f.read(12))
            if n==1 and kind in (3,4):tags[tag]=struct.unpack(order+('H' if kind==3 else 'I'),raw[:2 if kind==3 else 4])[0]
            else:tags[tag]=dict(type=kind,count=n)
    return dict(width=tags[256],height=tags[257],compression=tags.get(259),samples=tags.get(277),
                has_geotags=any(k in tags for k in (33550,33922,34264,34735)),tags=tags)

def parse_jucker_r(text):
    match=re.search(r'pa\s*<-\s*c\((.*?)\)\s*\n',text,re.S)
    if not match:raise ValueError('Jucker R coefficient vector not found')
    values=[float(v) for v in re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?',match.group(1))]
    if len(values)!=100:raise ValueError(f'Expected25×4 coefficients; got{len(values)}')
    table={int(values[i]):dict(a=values[i+1],b=values[i+2],g=values[i+3]) for i in range(0,100,4)}
    if set(table)!=set(range(25)):raise ValueError('Unexpected biome numbering')
    return table

def jucker_diameter(height,crown,forest_code,table):
    def one(k):
        c=table[k];return c['a']*(height*crown)**c['b']*math.exp(c['g']**2/2)
    if height<=0 or crown<=0:return float('nan'),'invalid'
    if str(forest_code)=='1':return one(21),'21'
    if str(forest_code)=='2':return one(20),'20'
    return (one(20)+one(21))/2,'mean(20,21)'

class KoreanAllometry:
    def __init__(self,equations,factors,fractions):
        self.eq={(r['species'],r['form'],r['part']):r for r in equations}
        self.factors={r['species']:r for r in factors}
        self.cf={r['species']:float(r['CF']) for r in fractions}
        self.dec=[s for s,r in self.factors.items() if r['forest_type']=='낙엽활엽수']
        self.con=[s for s,r in self.factors.items() if r['forest_type']=='침엽수']
        self.pines=['강원지방소나무','중부지방소나무']
        self.oaks=['상수리나무','굴참나무','신갈나무','졸참나무']
        assert len(self.factors)==22 and len(self.eq)==176
    def mapping(self,species,dbh,forest_code='',origin='field'):
        aliases={'낙엽송':'일본잎갈나무','편백나무':'편백','개서어나무':'서어나무','소사나무':'서어나무','백합나무':'튤립나무'}
        if species=='등나무':return [],'등나무 제외'
        if species=='소나무':return [('강원지방소나무' if dbh>40 else '중부지방소나무',1.)],'소나무40cm 규칙'
        if species in ('떡갈나무','기타참나무류'):targets=self.oaks;rule='참나무4종 예측량 평균'
        elif aliases.get(species,species) in self.factors:targets=[aliases.get(species,species)];rule='직접/README 별칭'
        elif origin=='field':
            targets=self.con if species in ('노간주나무','기타침엽수') else self.dec
            rule='기타 침엽수 평균' if targets is self.con else '기타 낙엽활엽수 평균'
        elif str(forest_code)=='1':targets=self.pines;rule='수종 미대응→소나무류 평균'
        elif str(forest_code)=='2':targets=self.dec;rule='수종 미대응→낙엽활엽수 평균'
        else:
            return [(s,.5/len(self.pines)) for s in self.pines]+[(s,.5/len(self.dec)) for s in self.dec], '혼효/임상미상→소나무류·낙엽활엽수 그룹 평균'
        return [(s,1/len(targets)) for s in targets],rule
    def predict(self,species,dbh,forest_code='',origin='field'):
        ws,rule=self.mapping(species,dbh,forest_code,origin)
        if not ws:return dict(AGB_kg=0.,carbon_kg=0.,carbon_cf_kg=0.,formula='excluded',mapping=rule,range_flag='',CF=.5,CF_fallback_weight=0.)
        totals={}; flags=set(); descriptions=[]; species_agb={s:0. for s,w in ws}
        for part in ('줄기','가지','잎'):
            totals[part]=0.
            for s,w in ws:
                r=self.eq[s,'a*D^b',part]
                predicted=float(r['a'])*dbh**float(r['b'])
                totals[part]+=w*predicted;species_agb[s]+=predicted
                descriptions.append(f'{s}:{part}:{r["a"]}*D^{r["b"]}:w={w:.8g}:p{r["page"]}')
                if not float(r['DBH_min_cm'])<=dbh<=float(r['DBH_max_cm']):flags.add(s)
        agb=sum(totals.values());carbon_cf=sum(w*species_agb[s]*self.cf.get(s,.5) for s,w in ws)
        cf=carbon_cf/agb if agb>0 else .5
        return dict(AGB_kg=agb,stem_kg=totals['줄기'],branch_kg=totals['가지'],leaf_kg=totals['잎'],
                    carbon_kg=agb*.5,carbon_cf_kg=carbon_cf,CF=cf,
                    CF_fallback_weight=sum(w for s,w in ws if s not in self.cf),mapping=rule,
                    range_flag=';'.join(sorted(flags)),formula=' | '.join(descriptions),
                    target_species=';'.join(f'{s}:{w:.8g}' for s,w in ws))

def training_ratio(reference,estimated):
    good=[(float(a),float(b)) for a,b in zip(reference,estimated) if math.isfinite(float(a)) and math.isfinite(float(b))]
    denom=sum(b for a,b in good)
    return sum(a for a,b in good)/denom if denom>0 else float('nan')

def dimensional_checks():
    assert 1000/1000*10000/400*.5==12.5
    assert math.isclose(training_ratio([10,30],[5,15]),2)
    assert math.isnan(training_ratio([10],[0]))
    assert all(math.isclose(a,b) for a,b in zip(sheet_bounds('35913003'),(129.05,35.225,129.075,35.25)))

## 04. Colab 처리 함수

기존NEON모듈을내장합니다.새자료다운로드·3차결과재계산셀은없으며추론·수고·CD부분만호출합니다.

In [ ]:
NEON_PIPELINE_SOURCE='"""Reusable inference/evaluation functions. Requires Colab scientific dependencies.\nCall configure(config, output_directory, projected_crs) before inference.\nDBH inputs: H, CD in metres. AGB input DBH in centimetres; output kilograms.\n"""\nfrom pathlib import Path\nimport inspect\nimport math\nimport json, importlib.metadata\nimport numpy as np\nimport pandas as pd\nimport geopandas as gpd\nimport rasterio\nfrom rasterio.merge import merge\nfrom rasterio.mask import mask\nfrom shapely.geometry import Point, box, Polygon\nfrom shapely.ops import unary_union\nfrom scipy.optimize import linear_sum_assignment\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.linear_model import LinearRegression\nimport matplotlib.pyplot as plt\nfrom scipy.ndimage import gaussian_filter, maximum_filter, label as ndi_label\nfrom skimage.segmentation import watershed\nfrom rasterio.features import shapes as raster_shapes\nfrom shapely.geometry import shape\nAUDIT=[]\ndef configure(config, output_directory, projected_crs):\n    global CFG, OUT, WORK_CRS, default\n    CFG=config\n    OUT=Path(output_directory)\n    OUT.mkdir(parents=True,exist_ok=True)\n    WORK_CRS=projected_crs\n    default=config[\'agb_default\']\ndef audit(stage,item,reason):\n    AUDIT.append(dict(stage=stage,item=str(item),reason=str(reason)))\n\n\ndef require_columns(df, cols, label):\n    missing = set(cols) - set(df.columns)\n    if missing:\n        raise ValueError(f\'{label} 필드 누락: {missing}; 실제 필드={list(df.columns)}\')\n\ndef offset_xy(e, n, distance, azimuth_deg):\n    rad = np.deg2rad(azimuth_deg)\n    return (e + distance * np.sin(rad), n + distance * np.cos(rad))\n\ndef estimate_dbh(H_m, CD_m, coefficients=None):\n    c = coefficients or CFG[\'jucker\']\n    H = np.asarray(H_m, dtype=float)\n    CD = np.asarray(CD_m, dtype=float)\n    return np.where((H > 0) & (CD > 0), c[\'a\'] * (H * CD) ** c[\'b\'] * np.exp(c.get(\'g\', 0) ** 2 / 2), np.nan)\n\ndef estimate_agb(dbh_cm, beta0=None, beta1=None):\n    b0 = default[\'beta0\'] if beta0 is None else beta0\n    b1 = default[\'beta1\'] if beta1 is None else beta1\n    D = np.asarray(dbh_cm, dtype=float)\n    with np.errstate(invalid=\'ignore\', divide=\'ignore\'):\n        return np.where(D > 0, np.exp(b0 + b1 * np.log(D)), np.nan)\n\ndef kg_to_t_ha(kg, area_m2):\n    return np.asarray(kg) / 1000 * 10000 / np.asarray(area_m2)\n\ndef carbon_kg(agb_kg, fraction=0.5):\n    return np.asarray(agb_kg) * fraction\n\ndef tile_coordinates(centers, buffer_m=50):\n    tiles = set()\n    for x, y in centers:\n        for e in range(int((x - buffer_m) // 1000), int((x + buffer_m) // 1000) + 1):\n            for n in range(int((y - buffer_m) // 1000), int((y + buffer_m) // 1000) + 1):\n                tiles.add((e * 1000, n * 1000))\n    return sorted(tiles)\n\ndef product_site_entry(product, site):\n    records = product.get(\'data\', product).get(\'siteCodes\', [])\n    return next((r for r in records if r.get(\'siteCode\') == site), dict(siteCode=site, availableMonths=[], availableReleases=[]))\n\ndef select_aop_pair(rgb_entry, chm_entry, target_year, allow_adjacent=False, priority=(2022, 2020), include_provisional=True):\n    common = sorted(set(rgb_entry.get(\'availableMonths\', [])) & set(chm_entry.get(\'availableMonths\', [])))\n    years = [int(target_year)] + ([int(y) for y in priority] if allow_adjacent else [])\n    selected = next((y for y in years if any((m.startswith(str(y) + \'-\') for m in common))), None)\n    if selected is None:\n        raise ValueError(f\'RGB/CHM 공통 연월 없음: 요청 연도 {years}; 공통 월 {common}\')\n    months = [m for m in common if m.startswith(str(selected) + \'-\')]\n    provisional = False\n    for entry in [rgb_entry, chm_entry]:\n        stable = {m for r in entry.get(\'availableReleases\', []) if r.get(\'release\') != \'PROVISIONAL\' for m in r.get(\'availableMonths\', [])}\n        prov = {m for r in entry.get(\'availableReleases\', []) if r.get(\'release\') == \'PROVISIONAL\' for m in r.get(\'availableMonths\', [])}\n        provisional |= any((m in prov and m not in stable for m in months))\n    if provisional and (not include_provisional):\n        raise ValueError(\'선택 연도에 PROVISIONAL 자료가 있습니다. aop_include_provisional=True 필요; 다른 해로 숨겨 대체하지 않습니다.\')\n    return dict(imagery_year=selected, common_months=months, include_provisional=bool(provisional), provisional=bool(provisional), adjacent_imagery=selected != int(target_year))\n\ndef aop_tiff_files(folder):\n    return sorted((p for p in Path(folder).rglob(\'*\') if p.is_file() and p.suffix.lower() in [\'.tif\', \'.tiff\']))\n\ndef raster_index(folder, expected_resolution):\n    rows = []\n    for path in sorted(Path(folder).rglob(\'*\')):\n        if path.suffix.lower() not in [\'.tif\', \'.tiff\']:\n            continue\n        with rasterio.open(path) as ds:\n            if not ds.crs or not ds.crs.is_projected:\n                raise ValueError(f\'미터 투영 CRS 필요: {path}\')\n            if not np.allclose(ds.res, expected_resolution, atol=expected_resolution * 0.05):\n                audit(\'resolution\', path, f\'예상 {expected_resolution}; 실제 {ds.res}\')\n                continue\n            rows.append(dict(path=str(path), crs=ds.crs, geometry=box(*ds.bounds)))\n    return rows\n\ndef crop_mosaic(index, region, region_crs, dest, bands):\n    if dest.exists():\n        return dest\n    datasets = []\n    coverage = []\n    try:\n        for r in index:\n            geom = gpd.GeoSeries([region], crs=region_crs).to_crs(r[\'crs\']).iloc[0]\n            if r[\'geometry\'].intersects(geom):\n                datasets.append(rasterio.open(r[\'path\']))\n                coverage.append(r[\'geometry\'])\n        if not datasets:\n            raise ValueError(\'해당 위치를 덮는 타일 없음\')\n        crs = datasets[0].crs\n        if any((d.crs != crs for d in datasets)):\n            raise ValueError(\'mosaic 입력 CRS 불일치\')\n        geom = gpd.GeoSeries([region], crs=region_crs).to_crs(crs).iloc[0]\n        if not unary_union(coverage).buffer(0.2).covers(geom):\n            raise ValueError(\'조사구 주변 영상 coverage 불완전\')\n        data, trans = merge(datasets, bounds=geom.bounds, indexes=bands)\n        profile = datasets[0].profile.copy()\n        profile.update(driver=\'GTiff\', height=data.shape[1], width=data.shape[2], count=len(bands), transform=trans, compress=\'deflate\', tiled=True, blockxsize=256, blockysize=256)\n        with rasterio.open(dest, \'w\', **profile) as out:\n            out.write(data)\n    finally:\n        for d in datasets:\n            d.close()\n    return dest\n\ndef raster_covers_geometry(index, geometry, geometry_crs):\n    if not index:\n        return False\n    bounds = [gpd.GeoSeries([r[\'geometry\']], crs=r[\'crs\']).to_crs(geometry_crs).iloc[0] for r in index]\n    return unary_union(bounds).buffer(0.2).covers(geometry)\n\ndef zonal_max(path, geom, geom_crs):\n    with rasterio.open(path) as ds:\n        g = gpd.GeoSeries([geom], crs=geom_crs).to_crs(ds.crs).iloc[0]\n        try:\n            arr, _ = mask(ds, [g], crop=True, filled=False, indexes=1)\n        except ValueError:\n            return np.nan\n        v = arr.compressed()\n        v = v[np.isfinite(v) & (v >= 0)]\n        return float(v.max()) if len(v) else np.nan\n\ndef predict_crowns(m, rgb, chm, plot):\n    cache = OUT / f\'{plot.plotID}_raw_predictions.csv\'\n    if cache.exists():\n        raw = pd.read_csv(cache)\n    else:\n        params = inspect.signature(m.predict_tile).parameters\n        path_key = \'path\' if \'path\' in params else \'raster_path\'\n        args = {path_key: str(rgb), \'patch_size\': CFG[\'patch_size\'], \'patch_overlap\': CFG[\'patch_overlap\']}\n        if \'iou_threshold\' in params:\n            args[\'iou_threshold\'] = CFG[\'nms_iou\']\n        raw = m.predict_tile(**args)\n        if isinstance(raw, tuple):\n            raw = raw[0]\n        if raw is None or len(raw) == 0:\n            raw = pd.DataFrame(columns=[\'xmin\', \'ymin\', \'xmax\', \'ymax\', \'score\'])\n        raw.to_csv(cache, index=False)\n    require_columns(raw, [\'xmin\', \'ymin\', \'xmax\', \'ymax\', \'score\'], \'DeepForest predictions\')\n    rows = []\n    with rasterio.open(rgb) as ds:\n        trans = ds.transform\n        crs = ds.crs\n        for idx, r in raw[raw.score >= CFG[\'score_threshold\']].iterrows():\n            corners = [trans * (x, y) for x, y in [(r.xmin, r.ymin), (r.xmax, r.ymin), (r.xmax, r.ymax), (r.xmin, r.ymax)]]\n            geom = Polygon(corners)\n            geom = gpd.GeoSeries([geom], crs=crs).to_crs(WORK_CRS).iloc[0]\n            if not plot.geometry.covers(geom.centroid):\n                continue\n            h = zonal_max(chm, geom, WORK_CRS)\n            if not np.isfinite(h):\n                audit(\'CHM_nodata\', f\'{plot.plotID}_{idx}\', \'CHM 없음\')\n                continue\n            if h < CFG[\'min_height_m\']:\n                continue\n            width = (r.xmax - r.xmin) * abs(trans.a)\n            length = (r.ymax - r.ymin) * abs(trans.e)\n            rows.append(dict(det_id=f\'{plot.plotID}_{idx}\', plotID=plot.plotID, siteID=plot.siteID, field_year=plot.field_year, score=r.score, xmin_px=r.xmin, ymin_px=r.ymin, xmax_px=r.xmax, ymax_px=r.ymax, H_m=h, CD_m=(width + length) / 2, width_m=width, length_m=length, geometry=geom))\n    return rows\n\ndef label_visibility(f, rasters):\n    f = f.copy()\n    local = []\n    for _, r in f.iterrows():\n        local.append(zonal_max(rasters[r.plotID][1], r.geometry.buffer(CFG[\'visible_radius_m\']), WORK_CRS))\n    f[\'local_CHM_m\'] = local\n    canopy = f.get(\'canopyPosition\', pd.Series(\'\', index=f.index)).fillna(\'\').str.strip().str.lower()\n    f[\'visible_by_position\'] = canopy.isin([\'full sun\', \'open grown\', \'partially shaded\'])\n    f[\'visible_by_height\'] = f.field_H_m.ge(CFG[\'visible_height_ratio\'] * f.local_CHM_m) & f.local_CHM_m.gt(0)\n    f[\'visible\'] = f.visible_by_position | f.visible_by_height\n    return f\n\ndef match_one_to_one(det, truth):\n    if det.empty or truth.empty:\n        return pd.DataFrame(columns=[\'det_id\', \'individualID\', \'plotID\', \'distance_m\'])\n    d = det.reset_index(drop=True)\n    t = truth.reset_index(drop=True)\n    n, m = (len(d), len(t))\n    weights = np.zeros((n, m + n))\n    valid = np.zeros((n, m), dtype=bool)\n    sizes = pd.to_numeric(t[CFG[\'match_size\']], errors=\'coerce\').fillna(0).to_numpy()\n    rank = pd.Series(sizes).rank(method=\'dense\').to_numpy() / (m + 1)\n    for i, r in d.iterrows():\n        for j, s in t.iterrows():\n            if r.geometry.covers(s.geometry):\n                valid[i, j] = True\n                distance = r.geometry.centroid.distance(s.geometry)\n                weights[i, j] = (n + 1) * 10 + rank[j] + 1e-06 / (1 + distance)\n    ri, ci = linear_sum_assignment(-weights)\n    rows = []\n    for i, j in zip(ri, ci):\n        if j < m and valid[i, j]:\n            rows.append(dict(det_id=d.iloc[i].det_id, individualID=t.iloc[j].individualID, plotID=d.iloc[i].plotID, distance_m=d.iloc[i].geometry.centroid.distance(t.iloc[j].geometry)))\n    return pd.DataFrame(rows, columns=[\'det_id\', \'individualID\', \'plotID\', \'distance_m\'])\n\ndef grouped_dbh_regression(df, n_splits=5):\n    result = pd.Series(np.nan, index=df.index)\n    fold_id = pd.Series(np.nan, index=df.index)\n    good = df[[\'H_m\', \'CD_m\', \'field_DBH_cm\']].gt(0).all(axis=1)\n    good &= np.isfinite(df[[\'H_m\', \'CD_m\', \'field_DBH_cm\']]).all(axis=1)\n    z = df.loc[good]\n    groups = z.plotID\n    if groups.nunique() < 2:\n        audit(\'regression\', \'GroupKFold\', \'독립 조사구 2개 미만\')\n        return (result, fold_id, None)\n    X = np.log(z[[\'H_m\', \'CD_m\']].to_numpy())\n    y = np.log(z.field_DBH_cm.to_numpy())\n    folds = GroupKFold(n_splits=min(n_splits, groups.nunique()))\n    for fold, (tr, te) in enumerate(folds.split(X, y, groups)):\n        if len(tr) < 4:\n            audit(\'regression\', fold, \'학습 나무 4개 미만\')\n            continue\n        m = LinearRegression().fit(X[tr], y[tr])\n        smear = float(np.mean(np.exp(y[tr] - m.predict(X[tr]))))\n        result.loc[z.index[te]] = np.exp(m.predict(X[te])) * smear\n        fold_id.loc[z.index[te]] = fold\n    m = LinearRegression().fit(X, y)\n    fit = dict(intercept=float(m.intercept_), log_H=float(m.coef_[0]), log_CD=float(m.coef_[1]), smearing=float(np.mean(np.exp(y - m.predict(X)))), n=len(z), n_plots=groups.nunique())\n    return (result, fold_id, fit)\n\ndef errors(truth, pred):\n    a = np.asarray(truth, dtype=float)\n    b = np.asarray(pred, dtype=float)\n    ok = np.isfinite(a) & np.isfinite(b)\n    a = a[ok]\n    b = b[ok]\n    if not len(a):\n        return dict(n=0, MAE=np.nan, RMSE=np.nan, Bias=np.nan, r=np.nan)\n    e = b - a\n    r = np.corrcoef(a, b)[0, 1] if len(a) > 1 and np.std(a) > 0 and (np.std(b) > 0) else np.nan\n    return dict(n=len(a), MAE=float(np.mean(abs(e))), RMSE=float(np.sqrt(np.mean(e ** 2))), Bias=float(np.mean(e)), r=r)\n\ndef plot_biomass_totals(detections_all, truth_all, truth_coord, pairs, plot, carbon_fraction=0.5, carbon_alternative=0.47):\n\n    def number(value):\n        try:\n            return float(value)\n        except (ValueError, TypeError):\n            return float(\'nan\')\n\n    def sum_valid(rows, column):\n        values = [number(r.get(column)) for r in rows]\n        return sum(values) if all((math.isfinite(v) for v in values)) else float(\'nan\')\n    area = number(plot.get(\'reported_area_m2\'))\n    if not math.isfinite(area) or area <= 0:\n        area = number(plot.get(\'area_m2\'))\n    if not math.isfinite(area) or area <= 0:\n        raise ValueError(\'양수 조사면적 필요\')\n    scale = lambda value: value / 1000 * 10000 / area\n    by_id = {r[\'individualID\']: r for r in truth_all}\n    if len(by_id) != len(truth_all):\n        raise ValueError(\'조사구 참조 개체 중복\')\n    det_by_id = {r[\'det_id\']: r for r in detections_all}\n    paired_truth = {r[\'individualID\'] for r in pairs}\n    paired_det = {r[\'det_id\'] for r in pairs}\n    coord_ids = {r[\'individualID\'] for r in truth_coord}\n    if not paired_truth <= by_id.keys() or not paired_det <= det_by_id.keys():\n        raise ValueError(\'매칭 ID가 조사구 모집단에 없음\')\n    reference = sum_valid(truth_all, \'ref_AGB_kg\')\n    pred = sum_valid(detections_all, \'AGB_default_kg\')\n    coord_reference = sum_valid(truth_coord, \'ref_AGB_kg\')\n    missing_agb = sum((not math.isfinite(number(r.get(\'ref_AGB_kg\'))) for r in truth_all))\n    matched_ref = sum_valid([by_id[k] for k in paired_truth], \'ref_AGB_kg\')\n    matched_pred = sum_valid([det_by_id[k] for k in paired_det], \'AGB_default_kg\')\n    unmatched_pred = sum_valid([r for r in detections_all if r[\'det_id\'] not in paired_det], \'AGB_default_kg\')\n    missed_ref = sum_valid([r for r in truth_all if r[\'individualID\'] not in paired_truth], \'ref_AGB_kg\')\n    no_coord = sum_valid([r for r in truth_all if r[\'individualID\'] not in coord_ids], \'ref_AGB_kg\')\n    missed_coord = sum_valid([r for r in truth_coord if r[\'individualID\'] not in paired_truth], \'ref_AGB_kg\')\n    field_year = int(plot[\'field_year\'])\n    imagery_year = int(plot.get(\'imagery_year\', field_year))\n    row = dict(plotID=plot[\'plotID\'], siteID=plot[\'siteID\'], field_year=field_year, imagery_year=imagery_year, same_year=field_year == imagery_year, imagery_provisional=plot.get(\'imagery_provisional\', False), area_m2=area, reported_area_m2=number(plot.get(\'reported_area_m2\', area)), polygon_area_m2=number(plot.get(\'polygon_area_m2\', area)), area_difference_m2=number(plot.get(\'polygon_area_m2\', area)) - area, n_truth=len(truth_all), n_truth_coord=len(truth_coord), n_detected=len(detections_all), n_matched=len(pairs), n_field_before_coordinate_filter=len(truth_all), n_without_matching_coordinate=len(truth_all) - len(truth_coord), n_missing_ref_AGB=missing_agb, AGB_reference_coverage=1 - missing_agb / len(truth_all) if truth_all else 1.0, reference_complete=missing_agb == 0, coordinates_complete=len(truth_all) == len(truth_coord), AGB_est_t_ha=scale(pred), AGB_ref_t_ha=scale(reference), AGB_ref_coordonly_t_ha=scale(coord_reference), matched_error_t_ha=scale(matched_pred - matched_ref), unmatched_detection_t_ha=scale(unmatched_pred), missed_truth_t_ha=scale(missed_ref), unmapped_truth_t_ha=scale(no_coord), missed_mapped_truth_t_ha=scale(missed_coord))\n    row[\'C_est_tC_ha\'] = row[\'AGB_est_t_ha\'] * carbon_fraction\n    row[\'C_ref_tC_ha\'] = row[\'AGB_ref_t_ha\'] * carbon_fraction\n    row[\'C_ref_coordonly_tC_ha\'] = row[\'AGB_ref_coordonly_t_ha\'] * carbon_fraction\n    row[\'C_est_047_tC_ha\'] = row[\'AGB_est_t_ha\'] * carbon_alternative\n    row[\'C_ref_047_tC_ha\'] = row[\'AGB_ref_t_ha\'] * carbon_alternative\n    row[\'C_bias_tC_ha\'] = row[\'C_est_tC_ha\'] - row[\'C_ref_tC_ha\']\n    row[\'C_missed_tC_ha\'] = row[\'missed_truth_t_ha\'] * carbon_fraction\n    row[\'estimate_reference_ratio\'] = pred / reference if reference > 0 else float(\'nan\')\n    if math.isfinite(reference) and math.isfinite(pred):\n        assert math.isclose(pred - reference, matched_pred - matched_ref + unmatched_pred - missed_ref, rel_tol=1e-09, abs_tol=1e-06)\n    return row\n\ndef comparison_statistics(truth, pred):\n    result = errors(truth, pred)\n    a = np.asarray(truth, dtype=float)\n    b = np.asarray(pred, dtype=float)\n    ok = np.isfinite(a) & np.isfinite(b)\n    a = a[ok]\n    b = b[ok]\n    mean_ref = float(a.mean()) if len(a) else np.nan\n    mean_est = float(b.mean()) if len(b) else np.nan\n    result.update(mean_reference=mean_ref, mean_estimate=mean_est, estimate_reference_ratio=mean_est / mean_ref if mean_ref > 0 else np.nan)\n    return result\n\ndef finalize_detection_frame(records, method):\n    cols = DET_COLUMNS if \'DET_COLUMNS\' in globals() else [\'det_id\', \'plotID\', \'siteID\', \'field_year\', \'score\', \'H_m\', \'CD_m\', \'geometry\']\n    frame = gpd.GeoDataFrame(records, geometry=\'geometry\', crs=WORK_CRS) if len(records) else gpd.GeoDataFrame(columns=cols, geometry=\'geometry\', crs=WORK_CRS)\n    frame[\'method\'] = method\n    frame[\'DBH_jucker_cm\'] = estimate_dbh(frame.H_m, frame.CD_m)\n    frame[\'AGB_default_kg\'] = estimate_agb(frame.DBH_jucker_cm)\n    frame[\'C_default_kg\'] = carbon_kg(frame.AGB_default_kg, CFG[\'carbon_fraction\'])\n    return frame\n\ndef deepforest_patch_candidates(detector, rgb, chm, plot, patch_size, cache_dir):\n    saved = []\n    for obj in [detector.model, getattr(detector.model, \'model\', None)]:\n        if obj is not None and hasattr(obj, \'score_thresh\'):\n            saved.append((obj, \'score_thresh\', obj.score_thresh))\n            obj.score_thresh = 0.05\n    configs = []\n    for obj in [detector.config, getattr(detector.config, \'retinanet\', None)]:\n        if obj is None:\n            continue\n        try:\n            old = obj[\'score_thresh\']\n            obj[\'score_thresh\'] = 0.05\n            configs.append((obj, old))\n        except (KeyError, TypeError, AttributeError):\n            pass\n    if not saved and (not configs):\n        raise RuntimeError(\'DeepForest 내부 score_thresh를 0.05로 설정하지 못했습니다. 낮은 점수 탐색을 숨겨 진행하지 않습니다.\')\n    oldcfg = CFG.copy()\n    oldout = globals()[\'OUT\']\n    directory = Path(cache_dir) / f\'patch_{patch_size}\'\n    directory.mkdir(parents=True, exist_ok=True)\n    fingerprint = dict(rgb=str(rgb), rgb_bytes=Path(rgb).stat().st_size, patch_size=int(patch_size), patch_overlap=CFG[\'patch_overlap\'], nms_iou=CFG[\'nms_iou\'], min_score=0.1, model_version=importlib.metadata.version(\'deepforest\'), model=CFG[\'model_name\'], revision=CFG[\'model_revision\'])\n    stamp = directory / f\'{plot.plotID}_input.json\'\n    rawpath = directory / f\'{plot.plotID}_raw_predictions.csv\'\n    if rawpath.exists() and (not stamp.exists() or json.loads(stamp.read_text()) != fingerprint):\n        rawpath.unlink()\n    try:\n        CFG[\'patch_size\'] = int(patch_size)\n        CFG[\'score_threshold\'] = 0.1\n        globals()[\'OUT\'] = directory\n        rows = predict_crowns(detector, rgb, chm, plot)\n        stamp.write_text(json.dumps(fingerprint, indent=2))\n    finally:\n        CFG.clear()\n        CFG.update(oldcfg)\n        globals()[\'OUT\'] = oldout\n        for obj, key, value in saved:\n            setattr(obj, key, value)\n        for obj, value in configs:\n            obj[\'score_thresh\'] = value\n    for row in rows:\n        row[\'det_id\'] = f"{plot.plotID}_p{patch_size}_{row[\'det_id\'].split(\'_\')[-1]}"\n    return finalize_detection_frame(rows, f\'DF_patch_{patch_size}\')\n\ndef variable_window_peaks(height, valid, resolution_m, diameter_a=2.51, diameter_b=0.00901, min_height=3.0):\n    mask = valid & (height >= min_height)\n    cross = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)\n    if resolution_m > (diameter_a + diameter_b * min_height ** 2) / 2:\n        raise ValueError(\'해상도가 최소 가변 창 반지름보다 큽니다. 더 세밀한 CHM 또는 현지 창 계수를 사용하세요.\')\n    maxima = mask & (height >= maximum_filter(np.where(valid, height, -np.inf), footprint=cross, mode=\'constant\', cval=-np.inf))\n    components, count = ndi_label(maxima, structure=cross.astype(int))\n    candidates = []\n    for k in range(1, count + 1):\n        coords = np.argwhere(components == k)\n        order = np.lexsort((coords[:, 1], coords[:, 0], -height[coords[:, 0], coords[:, 1]]))\n        r, c = coords[order[0]]\n        H = float(height[r, c])\n        radius = (diameter_a + diameter_b * H * H) / 2\n        pixels = int(np.ceil(radius / resolution_m))\n        r0 = max(0, r - pixels)\n        r1 = min(height.shape[0], r + pixels + 1)\n        c0 = max(0, c - pixels)\n        c1 = min(height.shape[1], c + pixels + 1)\n        rr, cc = np.ogrid[r0:r1, c0:c1]\n        circle = ((rr - r) * resolution_m) ** 2 + ((cc - c) * resolution_m) ** 2 <= radius ** 2\n        neighborhood = height[r0:r1, c0:c1][circle & valid[r0:r1, c0:c1]]\n        if len(neighborhood) and H >= float(neighborhood.max()) - 1e-07:\n            candidates.append((int(r), int(c), H, float(radius * 2)))\n    return sorted(candidates, key=lambda x: (-x[2], x[0], x[1]))\n\ndef chm_watershed_crowns(chm_path, plot, sigma_m=1.0):\n    with rasterio.open(chm_path) as ds:\n        raw = ds.read(1, masked=True)\n        transform = ds.transform\n        source_crs = ds.crs\n        resolution = float(abs(transform.a))\n        if not np.isclose(abs(transform.a), abs(transform.e)):\n            raise ValueError(\'CHM 정사각형 픽셀 필요\')\n        rawdata = np.asarray(raw.astype(float).filled(np.nan), dtype=float)\n        valid = np.isfinite(rawdata) & (rawdata >= 0)\n        weight = gaussian_filter(valid.astype(float), sigma=sigma_m / resolution)\n        smooth = gaussian_filter(np.where(valid, rawdata, 0), sigma=sigma_m / resolution) / np.maximum(weight, 1e-12)\n        peaks = variable_window_peaks(smooth, valid, resolution, CFG[\'chm_window_a\'], CFG[\'chm_window_b\'], CFG[\'min_height_m\'])\n        markers = np.zeros(rawdata.shape, dtype=np.int32)\n        accepted = []\n        for r, c, H, diameter in peaks:\n            if rawdata[r, c] < CFG[\'min_height_m\']:\n                continue\n            k = len(accepted) + 1\n            markers[r, c] = k\n            accepted.append((k, r, c, float(rawdata[r, c]), diameter))\n        if not accepted:\n            return finalize_detection_frame([], \'B_CHM\')\n        segments = watershed(-smooth, markers, mask=valid & (rawdata >= CFG[\'min_height_m\']))\n        crown_parts = {}\n        for geom, value in raster_shapes(segments.astype(\'int32\'), mask=segments > 0, transform=transform):\n            crown_parts.setdefault(int(value), []).append(shape(geom))\n    rows = []\n    for k, r, c, H, diameter in accepted:\n        xy = transform * (c + 0.5, r + 0.5)\n        point = gpd.GeoSeries([Point(*xy)], crs=source_crs).to_crs(WORK_CRS).iloc[0]\n        if not plot.geometry.covers(point):\n            continue\n        polygon = gpd.GeoSeries([unary_union(crown_parts[k])], crs=source_crs).to_crs(WORK_CRS).iloc[0]\n        area = float(polygon.area)\n        if area <= 0:\n            continue\n        rows.append(dict(det_id=f\'{plot.plotID}_CHM_{k}\', plotID=plot.plotID, siteID=plot.siteID, field_year=plot.field_year, imagery_year=plot.imagery_year, score=1.0, H_m=H, CD_m=2 * np.sqrt(area / np.pi), crown_area_m2=area, crown_geometry_wkt=polygon.wkt, peak_x=point.x, peak_y=point.y, window_diameter_m=diameter, geometry=polygon.envelope))\n    return finalize_detection_frame(rows, \'B_CHM\')\n\ndef combine_boxes_and_peaks(boxes, chm, dedup_m=1.0):\n    rows = []\n    used = set()\n    for _, boxrow in boxes.sort_values([\'score\', \'det_id\'], ascending=[False, True]).iterrows():\n        candidates = chm[chm.apply(lambda p: boxrow.geometry.covers(Point(p.peak_x, p.peak_y)), axis=1)] if len(chm) else chm\n        had_seed = len(candidates) > 0\n        candidates = candidates[~candidates.det_id.isin(used)]\n        if had_seed and candidates.empty:\n            continue\n        row = boxrow.to_dict()\n        row[\'det_id\'] = \'C_\' + str(row[\'det_id\'])\n        if len(candidates):\n            peak = candidates.sort_values([\'H_m\', \'det_id\'], ascending=[False, True]).iloc[0]\n            used.add(peak.det_id)\n            row.update(H_m=peak.H_m, peak_x=peak.peak_x, peak_y=peak.peak_y)\n        else:\n            row.update(peak_x=boxrow.geometry.centroid.x, peak_y=boxrow.geometry.centroid.y)\n        point = Point(row[\'peak_x\'], row[\'peak_y\'])\n        if any((point.distance(Point(x[\'peak_x\'], x[\'peak_y\'])) < dedup_m for x in rows)):\n            continue\n        row[\'fusion_source\'] = \'DeepForest_box\'\n        rows.append(row)\n    for _, p in chm.iterrows():\n        point = Point(p.peak_x, p.peak_y)\n        if any((g.covers(point) for g in boxes.geometry)):\n            continue\n        if any((point.distance(Point(x[\'peak_x\'], x[\'peak_y\'])) < dedup_m for x in rows)):\n            continue\n        row = p.to_dict()\n        row[\'det_id\'] = \'C_\' + row[\'det_id\']\n        row[\'fusion_source\'] = \'outside_CHM_peak\'\n        rows.append(row)\n    return finalize_detection_frame(rows, \'C_fusion\')\n\ndef detection_counts(detections, truth, plot_ids):\n    tp = fp = fn = 0\n    for pid in plot_ids:\n        d = detections[detections.plotID == pid]\n        f = truth[truth.plotID == pid]\n        hits = len(match_one_to_one(d, f))\n        tp += hits\n        fp += len(d) - hits\n        fn += len(f) - hits\n    return dict(TP=tp, FP=fp, FN=fn, precision=tp / (tp + fp) if tp + fp else np.nan, recall=tp / (tp + fn) if tp + fn else np.nan, F1=2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan)\n\ndef nested_group_parameter_selection(candidates, plot_table, truth, n_splits=5):\n    ids = np.array(sorted(plot_table.plotID.unique()))\n    outer = GroupKFold(n_splits=min(n_splits, len(ids)))\n    predictions = []\n    selections = []\n    inner_rows = []\n    split_rows = []\n    keys = sorted(candidates)\n    for fold, (train, test) in enumerate(outer.split(ids, groups=ids)):\n        trainids = ids[train]\n        testids = ids[test]\n        assert set(trainids).isdisjoint(testids)\n        for pid in trainids:\n            split_rows.append(dict(outer_fold=fold, plotID=pid, role=\'train\'))\n        for pid in testids:\n            split_rows.append(dict(outer_fold=fold, plotID=pid, role=\'test\'))\n        inner = GroupKFold(n_splits=min(CFG[\'search_inner_folds\'], len(trainids)))\n        scores = {key: dict(TP=0, FP=0, FN=0) for key in keys}\n        for inner_fold, (itr, ival) in enumerate(inner.split(trainids, groups=trainids)):\n            assert set(trainids[itr]).isdisjoint(trainids[ival])\n            assert set(trainids[ival]).isdisjoint(testids)\n            for key in keys:\n                result = detection_counts(candidates[key], truth, trainids[ival])\n                inner_rows.append(dict(outer_fold=fold, inner_fold=inner_fold, patch_size=key[0], score_threshold=key[1], **result))\n                for k in [\'TP\', \'FP\', \'FN\']:\n                    scores[key][k] += result[k]\n\n        def ranking(key):\n            counts = scores[key]\n            den = 2 * counts[\'TP\'] + counts[\'FP\'] + counts[\'FN\']\n            f1 = 2 * counts[\'TP\'] / den if den else -1\n            return (-f1, key[0], key[1])\n        best = min(keys, key=ranking)\n        chosen = candidates[best][candidates[best].plotID.isin(testids)].copy()\n        chosen[\'outer_fold\'] = fold\n        chosen[\'selected_patch_size\'] = best[0]\n        chosen[\'selected_score_threshold\'] = best[1]\n        predictions.append(chosen)\n        selections.append(dict(outer_fold=fold, patch_size=best[0], score_threshold=best[1], train_plots=\';\'.join(trainids), test_plots=\';\'.join(testids), **detection_counts(chosen, truth, testids)))\n    oof = gpd.GeoDataFrame(pd.concat(predictions, ignore_index=True), geometry=\'geometry\', crs=WORK_CRS)\n    deployment_rows = []\n    for key in keys:\n        deployment_rows.append(dict(patch_size=key[0], score_threshold=key[1], **detection_counts(candidates[key], truth, ids)))\n    deployment = pd.DataFrame(deployment_rows).sort_values([\'F1\', \'patch_size\', \'score_threshold\'], ascending=[False, True, True])\n    return (oof, pd.DataFrame(selections), pd.DataFrame(inner_rows), pd.DataFrame(split_rows), deployment)\n\ndef evaluate_detection_method(det, plot_table, all_truth, coordinate_truth):\n    metric_rows = []\n    pair_tables = {}\n    joined = None\n    for population in [\'visible\', \'all\']:\n        pairs = []\n        for _, p in plot_table.iterrows():\n            d = det[det.plotID == p.plotID]\n            f = coordinate_truth[coordinate_truth.plotID == p.plotID]\n            if population == \'visible\':\n                f = f[f.visible]\n            match = match_one_to_one(d, f)\n            pairs.append(match)\n            metric_rows.append(dict(plotID=p.plotID, siteID=p.siteID, field_year=p.field_year, population=population, **detection_counts(d, f, [p.plotID])))\n        pair_tables[population] = pd.concat(pairs, ignore_index=True)\n    visible = pair_tables[\'visible\']\n    joined = visible.merge(det.drop(columns=\'geometry\'), on=[\'det_id\', \'plotID\'], validate=\'one_to_one\')\n    joined = joined.merge(coordinate_truth.drop(columns=[\'geometry\', \'plotID\', \'siteID\', \'field_year\']), on=\'individualID\', validate=\'one_to_one\')\n    joined[\'DBH_regression_oof_cm\'], joined[\'cv_fold\'], _ = grouped_dbh_regression(joined, CFG[\'cv_folds\'])\n    joined[\'AGB_regression_oof_kg\'] = estimate_agb(joined.DBH_regression_oof_cm)\n    joined[\'AGB_known_species_kg\'] = estimate_agb(joined.DBH_jucker_cm, joined.species_beta0, joined.species_beta1)\n    joined[\'species_effect_kg\'] = joined.AGB_known_species_kg - joined.AGB_default_kg\n    joined[\'species_effect_pct\'] = 100 * (joined.AGB_known_species_kg / joined.AGB_default_kg - 1)\n    plotrows = []\n    for _, p in plot_table.iterrows():\n        ds = det[det.plotID == p.plotID]\n        fa = all_truth[all_truth.plotID == p.plotID]\n        fc = coordinate_truth[coordinate_truth.plotID == p.plotID]\n        pairs = pair_tables[\'all\']\n        pairs = pairs[pairs.plotID == p.plotID]\n        plotrows.append(plot_biomass_totals(ds.to_dict(\'records\'), fa.to_dict(\'records\'), fc.to_dict(\'records\'), pairs.to_dict(\'records\'), p.to_dict(), CFG[\'carbon_fraction\'], CFG[\'carbon_alternative\']))\n    return dict(detections=det, matched=joined, metrics=pd.DataFrame(metric_rows), pairs=pair_tables, plots=pd.DataFrame(plotrows))\n\ndef method_comparison_rows(method_results, requested_sites):\n    rows = []\n    for method, result in method_results.items():\n        for site in [\'ALL\'] + list(requested_sites):\n            pc = result[\'plots\']\n            mt = result[\'matched\']\n            dm = result[\'metrics\']\n            if site != \'ALL\':\n                pc = pc[pc.siteID == site]\n                mt = mt[mt.siteID == site]\n                dm = dm[dm.siteID == site]\n            stats = comparison_statistics(pc.C_ref_tC_ha, pc.C_est_tC_ha)\n            row = dict(method=method, siteID=site, n_plots=len(pc), n_field=int(pc.n_truth.sum()), n_detected=int(pc.n_detected.sum()), n_carbon_plots=stats[\'n\'], C_est_mean_tC_ha=stats[\'mean_estimate\'], C_ref_mean_tC_ha=stats[\'mean_reference\'], estimate_reference_ratio=stats[\'estimate_reference_ratio\'], carbon_r=stats[\'r\'], carbon_RMSE_tC_ha=stats[\'RMSE\'], carbon_MAE_tC_ha=stats[\'MAE\'])\n            for pop in [\'visible\', \'all\']:\n                q = dm[dm.population == pop]\n                tp = q.TP.sum()\n                fp = q.FP.sum()\n                fn = q.FN.sum()\n                row[\'F1_\' + pop] = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan\n                row[\'precision_\' + pop] = tp / (tp + fp) if tp + fp else np.nan\n                row[\'recall_\' + pop] = tp / (tp + fn) if tp + fn else np.nan\n            for name, a, b in [(\'height_Bias_m\', \'field_H_m\', \'H_m\'), (\'crown_Bias_m\', \'field_CD_m\', \'CD_m\'), (\'DBH_Bias_cm\', \'field_DBH_cm\', \'DBH_jucker_cm\'), (\'individual_AGB_Bias_kg\', \'ref_AGB_kg\', \'AGB_default_kg\')]:\n                row[name] = errors(mt[a], mt[b])[\'Bias\']\n            rows.append(row)\n    return pd.DataFrame(rows)\n\ndef detection_algorithm_smoke_checks():\n    from types import SimpleNamespace\n    yy, xx = np.mgrid[:64, :64]\n    canopy = 20 * np.exp(-((xx - 18) ** 2 + (yy - 32) ** 2) / 8) + 30 * np.exp(-((xx - 46) ** 2 + (yy - 32) ** 2) / 8)\n    path = OUT / \'synthetic_CHM_smoke.tif\'\n    with rasterio.open(path, \'w\', driver=\'GTiff\', width=64, height=64, count=1, dtype=\'float32\', crs=WORK_CRS, transform=rasterio.transform.from_origin(0, 64, 1, 1), nodata=-9999) as ds:\n        ds.write(canopy.astype(\'float32\'), 1)\n    p = SimpleNamespace(plotID=\'SMOKE\', siteID=\'SMOKE\', field_year=2021, imagery_year=2021, geometry=box(0, 0, 64, 64))\n    result = chm_watershed_crowns(path, p, 1.0)\n    assert len(result) == 2, \'합성 두 나무의 꼭대기 2개를 찾아야 합니다\'\n    assert np.allclose(sorted(result.H_m), [20, 30], atol=0.01)\n    assert np.allclose(result.CD_m, 2 * np.sqrt(result.crown_area_m2 / np.pi))\n    assert np.isclose(result.crown_area_m2.sum(), np.count_nonzero(canopy >= CFG[\'min_height_m\']))\n    first_peak = result.iloc[0].to_dict()\n    duplicate = dict(first_peak)\n    duplicate[\'det_id\'] = \'duplicate_box\'\n    duplicate[\'score\'] = 0.8\n    first_peak[\'score\'] = 0.9\n    boxes = gpd.GeoDataFrame([first_peak, duplicate], geometry=\'geometry\', crs=WORK_CRS)\n    fused = combine_boxes_and_peaks(boxes, result, 1.0)\n    assert len(fused) == 2, \'동일 꼭대기 중복 상자는 제거하고 상자 밖 두 번째 꼭대기는 추가해야 합니다\'\n    ids = [\'P0\', \'P1\', \'P2\', \'P3\']\n    drows = []\n    frows = []\n    for i, pid in enumerate(ids):\n        x = i * 20\n        drows.append(dict(det_id=\'d\' + pid, plotID=pid, score=0.8, H_m=20, CD_m=5, geometry=box(x - 3, -3, x + 3, 3)))\n        frows.append(dict(individualID=\'f\' + pid, plotID=pid, stemDiameter=30, geometry=Point(x, 0)))\n    ds = gpd.GeoDataFrame(drows, geometry=\'geometry\', crs=WORK_CRS)\n    fs = gpd.GeoDataFrame(frows, geometry=\'geometry\', crs=WORK_CRS)\n    plotdf = pd.DataFrame({\'plotID\': ids})\n    grid = {(300, 0.1): ds, (400, 0.2): ds.iloc[:2].copy()}\n    _, first, _, _, _ = nested_group_parameter_selection(grid, plotdf, fs, n_splits=2)\n    changed = fs.copy()\n    changed.loc[changed.plotID == \'P0\', \'geometry\'] = Point(9999, 9999)\n    _, second, _, _, _ = nested_group_parameter_selection(grid, plotdf, changed, n_splits=2)\n    fold = first[first.test_plots.str.split(\';\').apply(lambda z: \'P0\' in z)].index[0]\n    assert first.loc[fold, [\'patch_size\', \'score_threshold\']].equals(second.loc[fold, [\'patch_size\', \'score_threshold\']]), \'바깥 평가 정답이 선택에 새어 들어갔습니다\'\n    for _, r in first.iterrows():\n        assert set(r.train_plots.split(\';\')).isdisjoint(r.test_plots.split(\';\'))\n    path.unlink(missing_ok=True)\n    print(\'합성 CHM 꼭대기·watershed 면적·CD, GroupKFold 정답 격리 검증 통과\')\n\ndef plot_carbon_scatter(plot_table, output_folder):\n    for (year, image_year), cohort in plot_table.groupby([\'field_year\', \'imagery_year\']):\n        z = cohort.replace([np.inf, -np.inf], np.nan).dropna(subset=[\'C_ref_tC_ha\', \'C_est_tC_ha\'])\n        fig, ax = plt.subplots(figsize=(6, 6))\n        for site, q in z.groupby(\'siteID\'):\n            ax.scatter(q.C_ref_tC_ha, q.C_est_tC_ha, s=35, alpha=0.8, label=site, color={\'GRSM\': \'tab:blue\', \'ORNL\': \'tab:orange\'}.get(site))\n        if len(z):\n            hi = max(z[[\'C_ref_tC_ha\', \'C_est_tC_ha\']].max()) * 1.05\n            hi = max(hi, 1.0)\n            ax.plot([0, hi], [0, hi], \'k--\', label=\'1:1\')\n            ax.set_xlim(0, hi)\n            ax.set_ylim(0, hi)\n        ax.set(xlabel=\'Field-based reference carbon (tC/ha)\', ylabel=\'Estimated carbon (tC/ha)\', title=f\'Plot carbon: field {year} / imagery {image_year} (n={len(z)})\')\n        ax.legend()\n        ax.set_aspect(\'equal\', adjustable=\'box\')\n        filename = f\'plot_carbon_{year}.png\' if year == image_year else f\'plot_carbon_field{year}_imagery{image_year}.png\'\n        fig.savefig(Path(output_folder) / filename, dpi=170, bbox_inches=\'tight\')\n        plt.close(fig)\n\ndef build_site_summary(plot_table, detection_metrics, matched_table, requested_sites):\n    rows = []\n    for site in requested_sites:\n        site_plots = plot_table[plot_table.siteID == site]\n        if site_plots.empty:\n            rows.append(dict(siteID=site, n_plots=0, comparison_note=\'분석 결과 없음\'))\n            continue\n        for (field_year, image_year), q in site_plots.groupby([\'field_year\', \'imagery_year\']):\n            ids = set(q.plotID)\n            dm = detection_metrics[detection_metrics.plotID.isin(ids)]\n            mt = matched_table[matched_table.plotID.isin(ids)]\n            row = dict(siteID=site, field_year=field_year, imagery_year=image_year, comparison_note=\'영상 연도 = 실측 연도\' if field_year == image_year else f\'{site}은 영상 연도 ≠ 실측 연도\', n_plots=len(q), n_field_trees=int(q.n_truth.sum()), n_coord_trees=int(q.n_truth_coord.sum()), n_detections=int(q.n_detected.sum()))\n            for population, label in [(\'visible\', \'F1_visible\'), (\'all\', \'F1_all\')]:\n                z = dm[dm.population == population]\n                tp = z.TP.sum()\n                fp = z.FP.sum()\n                fn = z.FN.sum()\n                row[label] = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan\n            for label, refcol, predcol in [(\'height_Bias_m\', \'field_H_m\', \'H_m\'), (\'DBH_Bias_cm\', \'field_DBH_cm\', \'DBH_jucker_cm\'), (\'individual_AGB_Bias_kg\', \'ref_AGB_kg\', \'AGB_default_kg\')]:\n                row[label] = errors(mt[refcol], mt[predcol])[\'Bias\']\n            stats = comparison_statistics(q.C_ref_tC_ha, q.C_est_tC_ha)\n            row.update(n_carbon_plots=stats[\'n\'], C_est_mean_tC_ha=stats[\'mean_estimate\'], C_ref_mean_tC_ha=stats[\'mean_reference\'], estimate_reference_ratio=stats[\'estimate_reference_ratio\'])\n            rows.append(row)\n    return pd.DataFrame(rows)\n'
"""Colab scientific runtime, embedded in GEUMJEONGSAN_DeepForest_2021.ipynb.
Never imported/executed by the local Windows validation script.
"""
from contextlib import ExitStack
from types import SimpleNamespace
from xml.sax.saxutils import escape
import importlib.util, inspect, json, os, re, shutil, tempfile, time, warnings, gzip
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.merge import merge
from rasterio.enums import Resampling
from rasterio.plot import show
from rasterio.warp import reproject, transform_bounds
from rasterio.transform import from_origin
from rasterio.errors import NotGeoreferencedWarning
from shapely.geometry import Point, box, Polygon, shape
from shapely.ops import unary_union
from shapely import wkt
from pyproj import Transformer, CRS
from sklearn.model_selection import GroupKFold
import matplotlib.pyplot as plt
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CRS_WORK=CRS.from_epsg(5187)
TO_XY=Transformer.from_crs(4326,5187,always_xy=True)
TO_LL=Transformer.from_crs(5187,4326,always_xy=True)
JUCKER_URL='https://raw.githubusercontent.com/cran/itcSegment/master/R/dbh.R'
META_PREFIX='forests/v1/alsgedi_global_v6_float/'
META_BUCKET='https://dataforgood-fb-data.s3.amazonaws.com'
SCHEMA='geumjeongsan2021-v1'
AUDIT=[]

def audit(stage,item,message):
    AUDIT.append(dict(stage=stage,item=str(item),message=str(message)))

def json_save(path,obj):
    path=Path(path);tmp=path.with_name(path.name+'.part')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf-8');tmp.replace(path)

def geo_csv(frame,path):
    frame=frame.copy()
    if 'geometry' in frame:frame['geometry']=frame.geometry.map(lambda g:g.wkt if g is not None else '')
    pd.DataFrame(frame).to_csv(path,index=False)

def geo_read(path):
    df=pd.read_csv(path);df['geometry']=df.geometry.map(wkt.loads)
    return gpd.GeoDataFrame(df,geometry='geometry',crs=CRS_WORK)

def stat_signature(path):
    path=Path(path);s=path.stat()
    return dict(path=str(path),bytes=s.st_size,mtime_ns=s.st_mtime_ns)

def cache_valid(stamp,fingerprint,files):
    return all(Path(p).is_file() for p in files) and Path(stamp).is_file() and json.loads(Path(stamp).read_text())==fingerprint

def download(url,dest,optional=False):
    dest=Path(dest)
    if dest.exists() and dest.stat().st_size:return dest
    dest.parent.mkdir(parents=True,exist_ok=True);tmp=dest.with_name(dest.name+'.part')
    for attempt in range(5):
        try:
            with SESSION.get(url,stream=True,timeout=(30,180)) as response:
                if response.status_code==404 and optional:return None
                response.raise_for_status()
                with tmp.open('wb') as f:
                    for block in response.iter_content(1024*1024):
                        if block:f.write(block)
                if not response.headers.get('Content-Encoding') and response.headers.get('Content-Length'):
                    if tmp.stat().st_size!=int(response.headers['Content-Length']):raise IOError('전송 크기 불일치')
            tmp.replace(dest);return dest
        except Exception:
            if attempt==4:
                if optional:return None
                raise
            time.sleep(min(2**attempt,8))

def initialize():
    global DATA,OUT,CACHE,SESSION,NEON,JUCKER,ALLO,PLOTS,LAYERS,INPUT_SIGNATURE,MODEL
    MODEL=None
    DATA=ROOT/'gedi_geumjeongsan/data'
    if not DATA.exists() and (ROOT/'data').exists():DATA=ROOT/'data'
    needed=[DATA/'field'/n for n in ('seo2011_plots.csv','seo2011_trees.csv','seo2011_layers.csv')]
    needed += [DATA/'reference'/n for n in ('README.md','nifos2014_allometry.csv','nifos2014_carbon_factors.csv','nifos2010_carbon_fraction.csv')]
    missing=[str(p) for p in needed if not p.is_file()]
    if missing:raise FileNotFoundError('Drive 업로드 파일 누락:\n'+'\n'.join(missing))
    print((DATA/'reference/README.md').read_text(encoding='utf-8'))
    for module in (np,pd,gpd,rasterio):print(module.__name__,module.__version__)
    import importlib.metadata
    assert importlib.metadata.version('deepforest')=='2.1.0'
    SESSION=requests.Session();SESSION.mount('https://',HTTPAdapter(max_retries=Retry(total=5,backoff_factor=1,status_forcelist=(429,500,502,503,504))))
    CACHE=ROOT/'data/cache/geumjeongsan';CACHE.mkdir(parents=True,exist_ok=True)
    source=download(JUCKER_URL,CACHE/'reference/itcSegment_dbh.R')
    JUCKER=parse_jucker_r(source.read_text(encoding='utf-8'))
    for biome,expected in ((20,dict(a=.694,b=.73,g=0)),(21,dict(a=1.004,b=.73,g=0))):
        if JUCKER[biome]!=expected:warnings.warn(f'Jucker biome{biome} 원문값 변경: {JUCKER[biome]}; 원문값을 사용합니다.')
    ALLO=KoreanAllometry(read_csv_rows(DATA/'reference/nifos2014_allometry.csv'),read_csv_rows(DATA/'reference/nifos2014_carbon_factors.csv'),read_csv_rows(DATA/'reference/nifos2010_carbon_fraction.csv'))
    plot_table=pd.read_csv(DATA/'field/seo2011_plots.csv');assert len(plot_table)==50 and plot_table.plot_id.is_unique
    assert set(plot_table.zone)=={'중앙','동측','서측','남측','북측'} and plot_table.groupby('zone').size().eq(10).all()
    x,y=TO_XY.transform(plot_table['경도'].to_numpy(),plot_table['위도'].to_numpy())
    PLOTS=gpd.GeoDataFrame(plot_table,geometry=gpd.points_from_xy(x,y),crs=CRS_WORK)
    PLOTS['plotID']=PLOTS.plot_id;PLOTS['center_x']=x;PLOTS['center_y']=y
    PLOTS['area_m2']=400.;PLOTS['field_period']='2009-05..2010-09'
    LAYERS=pd.read_csv(DATA/'field/seo2011_layers.csv')
    t1=LAYERS[LAYERS['기호']=='T1'];assert len(t1)==50 and t1.plot_id.is_unique
    PLOTS['T1_min_m']=PLOTS.plot_id.map(t1.set_index('plot_id')['최소수고_m'])
    assert PLOTS.T1_min_m.notna().all()
    ortho=DATA/'ortho/2021'
    for sheet in SHEETS:
        path=ortho/f'(B060)정사영상_2021_{sheet}.tif';xmls=list(ortho.glob('*'+sheet+'.xml'))
        if not path.is_file() or len(xmls)!=1:raise FileNotFoundError(f'도엽{sheet} TIF/XML 업로드 필요')
        needed += [xmls[0]]
    # Pin the verified NEON resolution model; an optional uploaded provenance must agree.
    candidates=[ROOT/'neon_grsm_ornl/outputs_v4_resolution/resolution_model_provenance.json',ROOT/'resolution_model_provenance.json']
    for path in candidates:
        if path.exists():
            actual=json.loads(path.read_text())
            if actual['resolved_revision']!=HF_MODEL_PROVENANCE['resolved_revision']:raise ValueError('업로드한 HF 커밋과 내장 NEON 커밋이 다릅니다. 설정을 검토하세요.')
    reference_hashes={str(p.relative_to(DATA)):sha_file(p) for p in needed}
    spec=dict(schema=SCHEMA,code_sha=GEUM_CODE_SHA,files=reference_hashes,jucker_sha=sha_file(source),
              hf=HF_MODEL_PROVENANCE,neon_25cm=NEON_25CM_BASELINE,
              vectors=[stat_signature(p) for category in ('2025_임상도','등산로')
                       for p in sorted((DATA/'forest_map'/category).rglob('*')) if p.suffix.lower() in ('.shp','.dbf','.shx','.prj','.cpg')],
              ortho={s:stat_signature(ortho/f'(B060)정사영상_2021_{s}.tif') for s in SHEETS})
    INPUT_SIGNATURE=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()
    OUT=ROOT/'outputs'/('geumjeongsan_'+INPUT_SIGNATURE[:12]);OUT.mkdir(parents=True,exist_ok=True)
    (OUT/'figures').mkdir(exist_ok=True)
    json_save(OUT/'provenance.json',dict(spec,run_id=OUT.name,Jucker={k:JUCKER[k] for k in (20,21)},
               neon_pipeline_sha256=hashlib.sha256(NEON_PIPELINE_SOURCE.encode()).hexdigest(),
               units='D cm; H/CD m; AGB dry kg; aboveground C only; tC/ha',
               no_roots=True,field_period='2009-05..2010-09',image_year=2021))
    pd.DataFrame([dict(path=k,sha256=v) for k,v in reference_hashes.items()]).to_csv(OUT/'input_manifest.csv',index=False)
    # Self-contained notebook: no local module upload is required.
    module_dir=Path('/content/geumjeongsan_runtime');module_dir.mkdir(exist_ok=True)
    module_path=module_dir/'neon_pipeline.py';module_path.write_text(NEON_PIPELINE_SOURCE,encoding='utf-8')
    spec_module=importlib.util.spec_from_file_location('geum_neon_pipeline',module_path)
    NEON=importlib.util.module_from_spec(spec_module);spec_module.loader.exec_module(NEON)
    dimensional_checks()
    print('지상부만 계산; 뿌리 제외. 결과:',OUT,'\nHF 고정 커밋:',HF_MODEL_PROVENANCE['resolved_revision'])

def georeference():
    global ORTHO,AOI
    AOI=box(PLOTS.center_x.min()-80,PLOTS.center_y.min()-80,PLOTS.center_x.max()+80,PLOTS.center_y.max()+80)
    ortho=DATA/'ortho/2021';directory=CACHE/'vrt';directory.mkdir(exist_ok=True)
    ORTHO=[];rows=[]
    to5179=Transformer.from_crs(5187,5179,always_xy=True)
    for sheet in SHEETS:
        path=ortho/f'(B060)정사영상_2021_{sheet}.tif';xml_path=next(ortho.glob('*'+sheet+'.xml'))
        xml=ET.parse(xml_path).getroot()
        assert float(xml.findtext('.//지상표본거리'))==.25
        assert xml.findtext('.//투영원점')=='동부' and xml.findtext('.//기준타원체')=='GRS80'
        assert float(xml.findtext('.//Easting'))==200000 and float(xml.findtext('.//Northing'))==600000
        with warnings.catch_warnings():
            warnings.simplefilter('ignore',NotGeoreferencedWarning)
            with rasterio.open(path) as ds:
                width,height=ds.width,ds.height
                if ds.count<3 or ds.dtypes[0]!='uint8':raise ValueError('8bit RGB 필요')
        w,s,e,n=sheet_bounds(sheet)
        coordinates=[TO_XY.transform(lon,lat) for lon in (w,e) for lat in (s,n)]
        minx=min(x for x,y in coordinates);maxx=max(x for x,y in coordinates)
        miny=min(y for x,y in coordinates);maxy=max(y for x,y in coordinates)
        left,top=minx-50,maxy+50;right=left+width*.25;bottom=top-height*.25
        transform=from_origin(left,top,.25,.25)
        row=dict(sheet=sheet,width_px=width,height_px=height,gsd_m=.25,west=w,south=s,east=e,north=n,
                 left_5187=left,bottom_5187=bottom,right_5187=right,top_5187=top,
                 width_difference_m=width*.25-(maxx-minx+100),height_difference_m=height*.25-(maxy-miny+100),
                 source_crs='EPSG:5187 manual assignment',georef_method='sheet bbox +50m border, north-up',xml_sha=sha_file(xml_path))
        assert abs(row['width_difference_m'])<5 and abs(row['height_difference_m'])<5
        points=[to5179.transform(x,y) for x in (left,right) for y in (bottom,top)]
        mapped=(min(x for x,y in points),min(y for x,y in points),max(x for x,y in points),max(y for x,y in points))
        for name,value in zip(('left','bottom','right','top'),mapped):row[name+'_5179']=value
        if sheet=='35913003':
            errors=[a-b for a,b in zip(mapped,CONTROL_5179)]
            for name,err in zip(('left','bottom','right','top'),errors):row['control_error_'+name+'_m']=err
            row['control_max_abs_error_m']=max(map(abs,errors))
            assert row['control_max_abs_error_m']<=3,f'35913003 좌표 검증 실패: {errors}'
        # A VRT assigns metadata in the cache; the original TIFF is never modified.
        vrt=directory/(sheet+'.vrt')
        bands=''.join(f'''<VRTRasterBand dataType="Byte" band="{b}"><ColorInterp>{color}</ColorInterp><SimpleSource>
<SourceFilename relativeToVRT="0">{escape(str(path))}</SourceFilename><SourceBand>{b}</SourceBand>
<SrcRect xOff="0" yOff="0" xSize="{width}" ySize="{height}"/><DstRect xOff="0" yOff="0" xSize="{width}" ySize="{height}"/>
</SimpleSource></VRTRasterBand>''' for b,color in enumerate(('Red','Green','Blue'),1))
        vrt.write_text(f'<VRTDataset rasterXSize="{width}" rasterYSize="{height}"><SRS>EPSG:5187</SRS><GeoTransform>{left},0.25,0,{top},0,-0.25</GeoTransform>{bands}</VRTDataset>',encoding='utf-8')
        ORTHO.append(dict(sheet=sheet,path=path,vrt=vrt,geometry=box(left,bottom,right,top),transform=transform,width=width,height=height))
        rows.append(row)
    pd.DataFrame(rows).to_csv(OUT/'georef_check.csv',index=False);display(pd.DataFrame(rows))
    coverage=unary_union([o['geometry'] for o in ORTHO])
    for _,p in PLOTS.iterrows():
        if not coverage.buffer(.001).covers(box(p.center_x-60,p.center_y-60,p.center_x+60,p.center_y+60)):
            raise ValueError(f'{p.plot_id} 60m crop 전체를 덮는 영상이 없습니다. 조사구를 임의 제외하지 않습니다.')
    return pd.DataFrame(rows)

def read_vectors():
    global FOREST,TRAILS
    bounds5179=transform_bounds(5187,5179,*AOI.bounds,densify_pts=41)
    forest=[];trails=[]
    for category,target in (('2025_임상도',forest),('등산로',trails)):
        for district in ('금정구','북구','양산시'):
            paths=sorted((DATA/'forest_map'/category/district).glob('*.shp'))
            if not paths:raise FileNotFoundError(f'필수 shapefile 없음: {category}/{district}')
            for path in paths:
                df=gpd.read_file(path,bbox=bounds5179,encoding='cp949')
                print(category,district,'속성:',list(df.columns));display(df.drop(columns='geometry').head(3))
                if df.crs is None:
                    warnings.warn(f'{path.name} PRJ 없음: 입력 지시 EPSG5179 사용');df=df.set_crs(5179)
                if category=='2025_임상도':
                    if not ({'FRTP_NM','FRTP_CD'}&set(df)) or not ({'KOFTR_NM','KOFTR_GROU'}&set(df)):
                        raise ValueError('임상/수종 필드 확인 필요; 임의 필드로 진행하지 않습니다.')
                df=df.to_crs(CRS_WORK);df=df[df.geometry.notna()&~df.geometry.is_empty].copy()
                invalid=~df.geometry.is_valid
                if invalid.any():
                    audit('geometry_repair',path,f'{int(invalid.sum())}개 make_valid')
                    df.loc[invalid,'geometry']=df.loc[invalid,'geometry'].make_valid()
                df['source_district']=district;df['source_file']=path.name
                target.append(df[df.geometry.intersects(AOI)].copy())
    FOREST=gpd.GeoDataFrame(pd.concat(forest,ignore_index=True),geometry='geometry',crs=CRS_WORK)
    TRAILS=gpd.GeoDataFrame(pd.concat(trails,ignore_index=True),geometry='geometry',crs=CRS_WORK)
    assert len(FOREST)>0
    fields=pd.DataFrame([dict(layer=label,field=c) for label,df in [('forest',FOREST),('trails',TRAILS)] for c in df.columns])
    fields.to_csv(OUT/'vector_fields.csv',index=False)

def text(value):
    return '' if value is None or pd.isna(value) else str(value).strip()

SPECIES_CODES={'11':'소나무','12':'잣나무','13':'낙엽송','14':'리기다소나무','15':'곰솔','17':'편백나무','18':'삼나무',
 '10':'기타침엽수','31':'상수리나무','32':'신갈나무','33':'굴참나무','34':'기타참나무류','35':'오리나무',
 '36':'고로쇠나무','37':'자작나무','38':'박달나무','39':'밤나무','40':'물푸레나무','41':'서어나무',
 '42':'때죽나무','44':'백합나무','45':'포플러','46':'벚나무','47':'느티나무','49':'아까시나무','30':'기타활엽수','77':'침활혼효림','78':'죽림'}
SPECIES_CODES.update({'16':'잔나무','19':'가문비나무','20':'비자나무','21':'은행나무','43':'호두나무','48':'층층나무',
 '61':'가시나무','62':'구실잣밤나무','63':'녹나무','64':'굴거리나무','65':'황칠나무','66':'사스레피나무','67':'후박나무',
 '68':'새덕이','60':'기타상록활엽수','81':'미립목지','82':'제지','83':'관목덤불','91':'주거지','92':'초지','93':'경작지','94':'수체','95':'과수원','99':'기타'})

def forest_at(point):
    hits=list(FOREST.sindex.query(point,predicate='intersects'))
    def priority(i):
        r=FOREST.iloc[i];return (text(r.get('FRTP_CD')).removesuffix('.0') not in ('1','2','3'),('금정구','북구','양산시').index(r.source_district),int(i))
    if not hits:return dict(forest_code='',forest_type='임상도 밖/미피복',map_species='',forest_hit_count=0,forest_fallback=True)
    r=FOREST.iloc[min(hits,key=priority)]
    code=text(r.get('FRTP_CD')).removesuffix('.0');name=text(r.get('FRTP_NM'))
    if not code:
        code={'침엽수림':'1','활엽수림':'2','혼효림':'3','침활혼효림':'3','죽림':'4'}.get(name,'')
    code_species=text(r.get('KOFTR_GROU')).removesuffix('.0')
    species=text(r.get('KOFTR_NM')) or SPECIES_CODES.get(code_species,'')
    return dict(forest_code=code,forest_type=name or {'1':'침엽수림','2':'활엽수림','3':'혼효림','4':'죽림','0':'무립목지/비산림'}.get(code,'임상미상'),
                map_species=species,forest_hit_count=len(hits),forest_fallback=code not in ('1','2','3'),
                forest_source_district=r.source_district,forest_source_file=r.source_file)

def field_reference():
    rows=[];lookup=PLOTS.set_index('plot_id')
    for i,t in enumerate(read_csv_rows(DATA/'field/seo2011_trees.csv'),1):
        p=lookup.loc[t['plot_id']];d=float(t['흉고직경_cm']);h=float(t['수고_m'])
        included=t['수종']!='등나무' and d>=6
        estimate=ALLO.predict(t['수종'],d) if included else dict(AGB_kg=np.nan,carbon_kg=np.nan,carbon_cf_kg=np.nan,formula='excluded',mapping='등나무/DBH<6 제외',range_flag='',CF=.5)
        r=dict(tree_id=i,**t,zone=p.zone,DBH_cm=d,H_m=h,included_ge6=included,included_ge10=included and d>=10,
               exclusion_reason='등나무' if t['수종']=='등나무' else 'DBH<6cm' if d<6 else '',
               T1_min_m=p.T1_min_m,visible_by_T1=h>=p.T1_min_m,field_area_m2=400,**estimate)
        rows.append(r)
    result=pd.DataFrame(rows);assert len(result)==1750
    result.to_csv(OUT/'field_reference_trees.csv',index=False)
    return result

def rgb_crop(plot):
    directory=CACHE/'crops'/INPUT_SIGNATURE[:16]/plot.plot_id;directory.mkdir(parents=True,exist_ok=True)
    original=directory/'rgb_25cm.tif';expanded=directory/'rgb_10cm_bilinear.tif';stamp=directory/'rgb.json'
    region=box(plot.center_x-60,plot.center_y-60,plot.center_x+60,plot.center_y+60)
    selected=[r for r in ORTHO if r['geometry'].intersects(region)]
    fingerprint=dict(schema=SCHEMA,bounds=list(region.bounds),sources=[stat_signature(o['path']) for o in selected],
                     transforms=[list(o['transform']) for o in selected],resampling='bilinear to exact original extent at10cm')
    if not cache_valid(stamp,fingerprint,[original,expanded]):
        with ExitStack() as stack:
            sources=[stack.enter_context(rasterio.open(o['vrt'])) for o in selected]
            data,trans=merge(sources,bounds=region.bounds,res=.25,indexes=[1,2,3])
        assert data.shape==(3,480,480)
        profile=dict(driver='GTiff',height=480,width=480,count=3,dtype='uint8',crs=CRS_WORK,transform=trans,compress='deflate',tiled=True,blockxsize=256,blockysize=256)
        tmp=directory/'rgb_25cm.part.tif'
        with rasterio.open(tmp,'w',**profile) as ds:ds.write(data)
        tmp.replace(original)
        with rasterio.open(original) as ds:up=ds.read(out_shape=(3,1200,1200),resampling=Resampling.bilinear)
        profile.update(height=1200,width=1200,transform=trans*rasterio.Affine.scale(.4,.4))
        tmp=directory/'rgb_10cm.part.tif'
        with rasterio.open(tmp,'w',**profile) as ds:ds.write(up)
        tmp.replace(expanded);json_save(stamp,fingerprint)
    with rasterio.open(original) as lo,rasterio.open(expanded) as hi:
        assert lo.crs==hi.crs==CRS_WORK and lo.width==lo.height==480 and hi.width==hi.height==1200
        assert np.allclose(lo.res,.25) and np.allclose(hi.res,.1)
        assert np.allclose(lo.bounds,hi.bounds,atol=1e-8,rtol=0)
    return original,expanded,directory

def list_s3(prefix,delimiter=None,max_keys=100):
    params={'list-type':'2','prefix':prefix,'max-keys':max_keys}
    if delimiter:params['delimiter']=delimiter
    response=SESSION.get(META_BUCKET,params=params,timeout=60);response.raise_for_status()
    root=ET.fromstring(response.content)
    return [e.text for e in root.findall('.//{*}Contents/{*}Key')],[e.text for e in root.findall('.//{*}CommonPrefixes/{*}Prefix')]

def meta_sources():
    global META_SOURCES,META_METADATA,META_SIGNATURES
    bounds=transform_bounds(5187,4326,*AOI.bounds,densify_pts=41)
    quadkeys=quadkeys_for_bounds(bounds,9);print('금정산 AOI quadkey(zoom9):',quadkeys)
    sources={};metadata={};signatures={}
    search_dirs=[DATA/'chm',DATA/'raw/meta_chm',ROOT/'data/raw/meta_chm',ROOT/'gedi_hongcheon/data/raw/chm']+[Path(p) for p in EXTRA_CHM_DIRS]
    try:
        root_keys,prefixes=list_s3(META_PREFIX,delimiter='/')
        date_prefixes=[p for p in prefixes if any(k in p.lower().removeprefix(META_PREFIX) for k in ('date','geojson','metadata','footprint'))]
    except Exception as exc:
        root_keys=[];date_prefixes=[];audit('CHM_date_listing','S3',exc)
    for quad in quadkeys:
        cached=[p for d in search_dirs if d.exists() for p in d.rglob('*'+quad+'*.tif')]
        selected=None;target=box(*bounds).intersection(box(*quadkey_bounds(quad)))
        for candidate in sorted(cached):
            try:
                with rasterio.open(candidate) as ds:
                    footprint=box(*transform_bounds(ds.crs,4326,*ds.bounds,densify_pts=21))
                    if ds.crs.is_projected and max(ds.res)<=2 and footprint.buffer(1e-5).covers(target):
                        selected=candidate;break
            except Exception as exc:audit('CHM_cached_header',candidate,exc)
        sources[quad]=str(selected) if selected else META_BUCKET+'/'+META_PREFIX+'chm/'+quad+'.tif'
        if selected:signatures[quad]=stat_signature(selected)
        else:
            response=SESSION.head(sources[quad],timeout=60);response.raise_for_status()
            signatures[quad]=dict(url=sources[quad],etag=response.headers.get('ETag'),last_modified=response.headers.get('Last-Modified'),
                                  content_length=response.headers.get('Content-Length'))
        info=dict(quadkey=quad,source=sources[quad],date_status='unknown; no observation metadata found',date_files=[])
        candidates=[k for k in root_keys if quad in k and k.endswith(('.geojson','.geojson.gz','.json','.json.gz'))]
        for prefix in date_prefixes:
            try:
                keys,_=list_s3(prefix+quad,max_keys=20)
                candidates += [k for k in keys if k.endswith(('.geojson','.json','.geojson.gz','.json.gz'))]
            except Exception as exc:audit('CHM_date',quad,exc)
        for key in sorted(set(candidates)):
            path=download(META_BUCKET+'/'+key,CACHE/'meta_dates'/Path(key).name,optional=True)
            if path:
                info['date_files'].append(str(path));info['date_status']='associated metadata downloaded; acquisition fields checked per plot'
        metadata[quad]=info
    META_SOURCES=sources;META_METADATA=metadata;META_SIGNATURES=signatures
    for q in metadata:metadata[q]['source_signature']=signatures[q]
    json_save(OUT/'meta_chm_sources.json',metadata)
    return sources

def chm_crop(plot,directory):
    native=directory/'meta_chm_native.tif';local=directory/'meta_chm_5187_1m.tif';stamp=directory/'chm.json'
    region=box(plot.center_x-60,plot.center_y-60,plot.center_x+60,plot.center_y+60)
    fingerprint=dict(schema=SCHEMA,bounds=list(region.bounds),sources=META_SOURCES,source_signatures=META_SIGNATURES)
    if not cache_valid(stamp,fingerprint,[native,local]):
        # COG HTTP range windows only; never download a global quad in full.
        for attempt in range(5):
            try:
                with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',GDAL_HTTP_MAX_RETRY=5,GDAL_HTTP_RETRY_DELAY=2,VSI_CACHE=True):
                    with ExitStack() as stack:
                        opened=[(q,stack.enter_context(rasterio.open(path))) for q,path in META_SOURCES.items()]
                        crs=opened[0][1].crs
                        assert all(ds.crs==crs for q,ds in opened)
                        bounds=transform_bounds(CRS_WORK,crs,*region.bounds,densify_pts=41)
                        selected=[ds for q,ds in opened if box(*ds.bounds).intersects(box(*bounds))]
                        if not selected:raise ValueError('금정산 범위를 덮는 Meta CHM 없음')
                        coverage=unary_union([box(*ds.bounds) for ds in selected])
                        if not coverage.buffer(.01).covers(box(*bounds)):raise ValueError('CHM source coverage 불완전')
                        arr,trans=merge(selected,bounds=bounds,indexes=[1],nodata=np.nan,target_aligned_pixels=True)
                        tags={q:dict(tags=ds.tags(),band_tags=ds.tags(1),crs=ds.crs.to_string(),res=list(ds.res)) for q,ds in opened}
                        dtype=str(arr.dtype);assert np.issubdtype(arr.dtype,np.floating),'float형 Meta CHM 필요'
                break
            except Exception:
                if attempt==4:raise
                time.sleep(min(2**attempt,8))
        arr=arr.astype('float32');arr[(arr<0)|~np.isfinite(arr)]=np.nan
        profile=dict(driver='GTiff',count=1,height=arr.shape[1],width=arr.shape[2],crs=crs,transform=trans,dtype='float32',nodata=np.nan,compress='deflate')
        tmp=directory/'chm_native.part.tif'
        with rasterio.open(tmp,'w',**profile) as ds:ds.write(arr)
        tmp.replace(native)
        # Also deliver a local-metric 1m crop for inspection; inference uses native
        # pixels so zonal maximum H is not changed by CHM reprojection.
        out=np.full((120,120),np.nan,'float32');local_transform=from_origin(plot.center_x-60,plot.center_y+60,1,1)
        reproject(arr[0],out,src_transform=trans,src_crs=crs,dst_transform=local_transform,dst_crs=CRS_WORK,
                  src_nodata=np.nan,dst_nodata=np.nan,resampling=Resampling.max)
        profile.update(height=120,width=120,crs=CRS_WORK,transform=local_transform)
        tmp=directory/'chm_5187.part.tif'
        with rasterio.open(tmp,'w',**profile) as ds:ds.write(out,1)
        tmp.replace(local)
        json_save(directory/'chm_source_tags.json',tags);json_save(stamp,fingerprint)
    with rasterio.open(native) as ds:
        # Every comparison pixel must be valid (including valid zero height).
        geom=gpd.GeoSeries([plot.geometry.buffer(31)],crs=CRS_WORK).to_crs(ds.crs).iloc[0]
        from rasterio.mask import mask
        data,masked_transform=mask(ds,[geom],crop=True,filled=False,indexes=1)
        from rasterio.features import geometry_mask
        inside=geometry_mask([geom],out_shape=data.shape,transform=masked_transform,invert=True)
        valid=(~np.ma.getmaskarray(data))&np.isfinite(data.data)
        valid_fraction=float(np.count_nonzero(inside&valid)/np.count_nonzero(inside))
        if valid_fraction<.999:raise ValueError(f'{plot.plot_id} 30m 비교범위 CHM미피복 {100*(1-valid_fraction):.3f}%; 탄소0으로처리하지않습니다.')
    dates=[];file_dates=[]
    tag_file=directory/'chm_source_tags.json'
    if tag_file.exists():
        for quad,payload in json.loads(tag_file.read_text()).items():
            META_METADATA[quad]['native_raster_metadata']=payload
            for namespace in ('tags','band_tags'):
                for k,v in payload.get(namespace,{}).items():
                    if any(t in k.lower() for t in ('date','year','time')):
                        if any(t in k.lower() for t in ('acq','observ','image','capture','sensing')):dates.append(f'tag:{k}={v}')
                        else:file_dates.append(f'tag:{k}={v}')
    region_ll=gpd.GeoSeries([region],crs=CRS_WORK).to_crs(4326).iloc[0]
    for quad,info in META_METADATA.items():
        for path in info['date_files']:
            try:
                with Path(path).open('rb') as stream:is_gzip=stream.read(2)==b'\x1f\x8b'
                obj=json.loads(gzip.open(path,'rt',encoding='utf-8').read() if is_gzip else Path(path).read_text(encoding='utf-8'))
                for f in obj.get('features',[]):
                    if f.get('geometry') and not shape(f['geometry']).intersects(region_ll):continue
                    for k,v in f.get('properties',{}).items():
                        if any(t in k.lower() for t in ('date','year','time','acq')):
                            dates.append(f'{k}={v}')
            except Exception as exc:audit('CHM_date_parse',path,exc)
    info=dict(plot_id=plot.plot_id,native_path=str(native),local_5187_path=str(local),
              observation_dates=';'.join(sorted(set(dates))) or 'unknown',valid_fraction=valid_fraction,
              file_timestamps_not_acquisition=';'.join(sorted(set(file_dates))),
              height_extraction='maximum of native Meta pixels; 5187 1m crop is display-only')
    return native,local,info

def build_crops():
    global RASTERS,CHM_INFO
    RASTERS={};rows=[]
    meta_sources()
    for _,p in PLOTS.iterrows():
        low,high,directory=rgb_crop(p);chm,display_chm,info=chm_crop(p,directory)
        RASTERS[p.plot_id]=(low,high,chm);rows.append(dict(info,rgb_25cm=str(low),rgb_10cm=str(high)))
        print(p.plot_id,'crop 준비:',low.name,high.name,'CHM 날짜:',info['observation_dates'])
    CHM_INFO=pd.DataFrame(rows);CHM_INFO.to_csv(OUT/'raster_manifest.csv',index=False)
    json_save(OUT/'meta_chm_sources.json',META_METADATA)

def load_model():
    global MODEL
    if globals().get('MODEL') is not None:return MODEL
    from deepforest import main
    MODEL=main.deepforest()
    MODEL.load_model(model_name=HF_MODEL_PROVENANCE['model_name'],revision=HF_MODEL_PROVENANCE['resolved_revision'])
    changed=False
    for obj in (MODEL.model,getattr(MODEL.model,'model',None)):
        if obj is not None and hasattr(obj,'score_thresh'):obj.score_thresh=.05;changed=True
    for cfg in (MODEL.config,getattr(MODEL.config,'retinanet',None)):
        if cfg is None:continue
        try:previous=cfg['score_thresh'];cfg['score_thresh']=.05;changed=True
        except (TypeError,KeyError,AttributeError):pass
    if not changed:raise RuntimeError('NEON3차와같이내부score floor.05를설정하지못했습니다.임계값누락을숨기지않습니다.')
    json_save(OUT/'model_provenance.json',HF_MODEL_PROVENANCE)
    return MODEL

def estimate_boxes():
    frames=[]
    for _,p in PLOTS.iterrows():
        low,rgb,chm=RASTERS[p.plot_id]
        fp=dict(schema=SCHEMA,run_signature=INPUT_SIGNATURE,geometry=p.geometry.wkt,
                rgb_sha=sha_file(rgb),chm_sha=sha_file(chm),hf=HF_MODEL_PROVENANCE,score=.2,patch=600,min_height=3.)
        ident=hashlib.sha256(json.dumps(fp,sort_keys=True).encode()).hexdigest()[:16]
        directory=CACHE/'detections'/ident;directory.mkdir(parents=True,exist_ok=True)
        processed=directory/'detected.csv';stamp=directory/'complete.json'
        if cache_valid(stamp,fp,[processed]):
            frame=geo_read(processed);print(p.plot_id,'탐지 캐시 재사용');frames.append(frame);continue
        raw=directory/f'{p.plot_id}_raw_predictions.csv'
        raw_stamp=directory/'raw_complete.json'
        if raw.exists() and (not raw_stamp.exists() or json.loads(raw_stamp.read_text()).get('sha256')!=sha_file(raw)):
            raw.replace(raw.with_suffix('.incomplete.csv'))
        # The original NEON inference, center filter and native CHM maximum are reused.
        cfg=dict(score_threshold=.2,patch_size=600,patch_overlap=.25,nms_iou=.15,min_height_m=3.,
                 agb_default=dict(beta0=-2.2118,beta1=2.4133))
        NEON.configure(cfg,directory,CRS_WORK)
        plot=SimpleNamespace(plotID=p.plot_id,siteID='GEUMJEONGSAN',field_year='2009-2010',
                            geometry=box(p.center_x-60,p.center_y-60,p.center_x+60,p.center_y+60))
        detector=None if raw.is_file() else load_model()
        records=NEON.predict_crowns(detector,rgb,chm,plot)
        json_save(raw_stamp,dict(sha256=sha_file(raw)))
        rows=[]
        for row in records:
            attrs=forest_at(row['geometry'].centroid)
            d,biome=jucker_diameter(row['H_m'],row['CD_m'],attrs['forest_code'],JUCKER)
            value=ALLO.predict(attrs['map_species'],d,attrs['forest_code'],'map')
            r=dict(row,zone=p.zone,plot_id=p.plot_id,**attrs,DBH_cm=d,Jucker_biome=biome,**value)
            for delta,label in ((-.1,'height_minus10'),(.1,'height_plus10')):
                changed,_=jucker_diameter(row['H_m']*(1+delta),row['CD_m'],attrs['forest_code'],JUCKER)
                v=ALLO.predict(attrs['map_species'],changed,attrs['forest_code'],'map')
                r['carbon_'+label+'_kg']=v['carbon_kg'];r['DBH_'+label+'_cm']=changed
            rows.append(r)
        if rows:frame=gpd.GeoDataFrame(rows,geometry='geometry',crs=CRS_WORK)
        else:
            columns=['det_id','plotID','plot_id','zone','score','H_m','CD_m','DBH_cm','AGB_kg','carbon_kg','carbon_cf_kg',
                     'carbon_height_minus10_kg','carbon_height_plus10_kg','forest_type','forest_code','map_species','formula','mapping','range_flag','geometry']
            frame=gpd.GeoDataFrame(columns=columns,geometry='geometry',crs=CRS_WORK)
        geo_csv(frame,processed.with_suffix('.part.csv'));processed.with_suffix('.part.csv').replace(processed)
        json_save(stamp,fp);frames.append(frame)
        print(p.plot_id,'탐지:',len(frame))
    result=gpd.GeoDataFrame(pd.concat(frames,ignore_index=True),geometry='geometry',crs=CRS_WORK)
    geo_csv(result,OUT/'detected_trees.csv')
    return result

def metric(reference,estimate):
    a=np.asarray(reference,float);b=np.asarray(estimate,float);good=np.isfinite(a)&np.isfinite(b);a=a[good];b=b[good]
    if not len(a):return dict(n=0,reference_mean=np.nan,estimate_mean=np.nan,estimate_reference_ratio=np.nan,r=np.nan,RMSE=np.nan,MAE=np.nan,Bias=np.nan)
    delta=b-a
    return dict(n=len(a),reference_mean=float(a.mean()),estimate_mean=float(b.mean()),
                estimate_reference_ratio=float(b.mean()/a.mean()) if a.mean()>0 else np.nan,
                r=float(np.corrcoef(a,b)[0,1]) if len(a)>1 and np.std(a)>0 and np.std(b)>0 else np.nan,
                RMSE=float(np.sqrt(np.mean(delta**2))),MAE=float(np.mean(abs(delta))),Bias=float(delta.mean()))

def calibrate_zone_holdout(g):
    output=pd.DataFrame(index=g.index,columns=['C_calibrated_OOF_tC_ha','calibration_factor'],dtype=float)
    output['heldout_zone']='';records=[]
    assert g.zone.nunique()==5
    for fold,(train,test) in enumerate(GroupKFold(n_splits=5).split(g,groups=g.zone)):
        tr=g.iloc[train];te=g.iloc[test]
        assert set(tr.zone).isdisjoint(set(te.zone)) and te.zone.nunique()==1
        factor=training_ratio(tr.C_reference_tC_ha,tr.C_estimated_tC_ha)
        output.loc[te.index,'calibration_factor']=factor
        output.loc[te.index,'C_calibrated_OOF_tC_ha']=te.C_estimated_tC_ha*factor
        output.loc[te.index,'heldout_zone']=te.zone.iloc[0]
        records.append(dict(outer_fold=fold,train_zones=';'.join(sorted(set(tr.zone))),test_zone=te.zone.iloc[0],
                            train_plots=';'.join(tr.plot_id),test_plots=';'.join(te.plot_id),factor=factor,
                            n_train=len(tr),n_test=len(te),fit='sum(train_reference_density)/sum(train_estimated_density)'))
    return output,records

def compare_plots(detected,reference):
    rows=[]
    scenarios=[('square20','CF0.5','height0','carbon_kg'),('circle30','CF0.5','height0','carbon_kg'),
               ('square20','species_CF','height0','carbon_cf_kg'),('square20','CF0.5','height_minus10','carbon_height_minus10_kg'),
               ('square20','CF0.5','height_plus10','carbon_height_plus10_kg')]
    for _,p in PLOTS.iterrows():
        refs=reference[reference.plot_id==p.plot_id]
        boxes=detected[detected.plot_id==p.plot_id]
        for area_kind,cf,height,column in scenarios:
            geometry=box(p.center_x-10,p.center_y-10,p.center_x+10,p.center_y+10) if area_kind=='square20' else p.geometry.buffer(30,quad_segs=128)
            area=400. if area_kind=='square20' else math.pi*30**2
            chosen=boxes[boxes.geometry.centroid.map(geometry.covers).astype(bool)].copy()
            for threshold,include in ((10,'included_ge10'),(6,'included_ge6')):
                fr=refs[refs[include]];refcol='carbon_cf_kg' if cf=='species_CF' else 'carbon_kg'
                cref=float(fr[refcol].sum())/1000/.04
                cest=float(chosen[column].sum())/1000*10000/area
                visible=fr[fr.visible_by_T1];hidden=fr[~fr.visible_by_T1]
                center=forest_at(p.geometry)
                formula=' | '.join(sorted(set(chosen.formula)))
                r=dict(plot_id=p.plot_id,zone=p.zone,area_kind=area_kind,area_m2=area,reference_area_m2=400,
                       CF_scenario=cf,height_scenario=height,field_DBH_min_cm=threshold,
                       C_reference_tC_ha=cref,C_estimated_tC_ha=cest,estimate_reference_ratio=cest/cref if cref>0 else np.nan,
                       n_detected=len(chosen),n_detected_per400m2=len(chosen)*400/area,
                       n_field=len(fr),n_visible=len(visible),n_occluded=len(hidden),
                       C_visible_tC_ha=float(visible[refcol].sum())/1000/.04,C_occluded_tC_ha=float(hidden[refcol].sum())/1000/.04,
                       visible_carbon_share=float(visible[refcol].sum()/fr[refcol].sum()) if fr[refcol].sum()>0 else np.nan,
                       occluded_carbon_share=float(hidden[refcol].sum()/fr[refcol].sum()) if fr[refcol].sum()>0 else np.nan,
                       forest_type=center['forest_type'],map_species=center['map_species'],used_formula=formula,
                       detected_range_extrapolation_count=int(chosen.range_flag.fillna('').ne('').sum()),
                       detected_below10_count=int((chosen.DBH_cm<10).sum()),
                       C_estimated_DBH10only_tC_ha=float(chosen.loc[chosen.DBH_cm>=10,column].sum())/1000*10000/area,
                       reference_proxy_for_circle=area_kind=='circle30',T1_min_m=p.T1_min_m)
                rows.append(r)
    result=pd.DataFrame(rows)
    assert len(result)==50*5*2 and result.groupby(['area_kind','CF_scenario','height_scenario','field_DBH_min_cm']).size().eq(50).all()
    result['C_calibrated_OOF_tC_ha']=np.nan;result['calibration_factor']=np.nan;result['heldout_zone']=''
    folds=[]
    keys=['area_kind','CF_scenario','height_scenario','field_DBH_min_cm']
    for key,g in result.groupby(keys,sort=False):
        calibrated,records=calibrate_zone_holdout(g)
        for col in calibrated:result.loc[g.index,col]=calibrated[col]
        folds.extend(dict(zip(keys,key),**record) for record in records)
    result.to_csv(OUT/'plot_comparison_sensitivity.csv',index=False)
    primary=baseline_rows(result)
    primary['circle30_reference_is_20m_proxy']=True
    primary['C_reference_ge6_tC_ha']=primary.plot_id.map(result[(result.area_kind=='square20')&(result.CF_scenario=='CF0.5')&(result.height_scenario=='height0')&(result.field_DBH_min_cm==6)].set_index('plot_id').C_reference_tC_ha)
    for label,area_kind,cf,height in [('circle30','circle30','CF0.5','height0'),('speciesCF','square20','species_CF','height0'),
                                     ('height_minus10','square20','CF0.5','height_minus10'),('height_plus10','square20','CF0.5','height_plus10')]:
        scenario=result[(result.area_kind==area_kind)&(result.CF_scenario==cf)&(result.height_scenario==height)&(result.field_DBH_min_cm==10)].set_index('plot_id')
        primary['C_estimated_'+label+'_tC_ha']=primary.plot_id.map(scenario.C_estimated_tC_ha)
    primary.to_csv(OUT/'plot_comparison.csv',index=False)
    pd.DataFrame(folds).to_csv(OUT/'calibration_folds.csv',index=False)
    summaries=[]
    for key,g in result.groupby(keys,sort=False):
        for zone,q in [('ALL',g)]+list(g.groupby('zone')):
            a=metric(q.C_reference_tC_ha,q.C_estimated_tC_ha);b=metric(q.C_reference_tC_ha,q.C_calibrated_OOF_tC_ha)
            summaries.append(dict(zip(keys,key),zone=zone,source='금정산',**a,
                calibrated_RMSE=b['RMSE'],calibrated_MAE=b['MAE'],calibrated_ratio=b['estimate_reference_ratio'],calibrated_r=b['r'],
                n_detected=int(q.n_detected.sum()),n_visible=int(q.n_visible.sum()),n_occluded=int(q.n_occluded.sum()),F1_visible=np.nan))
    summaries.append(dict(source='NEON25cm 기준선(보정 미사용)',zone='ALL',area_kind='NEON original footprint',CF_scenario='CF0.5',
        height_scenario='height0',field_DBH_min_cm=10,n=23,estimate_reference_ratio=NEON_25CM_BASELINE['estimate_reference_ratio'],
        F1_visible=NEON_25CM_BASELINE['F1_visible'],RMSE=NEON_25CM_BASELINE['carbon_RMSE_tC_ha'],
        reference_mean=NEON_25CM_BASELINE['reference_mean'],estimate_mean=NEON_25CM_BASELINE['estimate_mean'],
        r=NEON_25CM_BASELINE['r'],MAE=NEON_25CM_BASELINE['MAE']))
    summary=pd.DataFrame(summaries);summary.to_csv(OUT/'summary_sensitivity.csv',index=False)
    core=summary[(summary.source!='금정산')|((summary.area_kind=='square20')&(summary.CF_scenario=='CF0.5')&(summary.height_scenario=='height0')&(summary.field_DBH_min_cm==10))]
    assert len(core)==7
    core.to_csv(OUT/'summary.csv',index=False)
    return result,summary

def baseline_rows(comparison):
    return comparison[(comparison.area_kind=='square20')&(comparison.CF_scenario=='CF0.5')&(comparison.height_scenario=='height0')&(comparison.field_DBH_min_cm==10)].copy()

def figures(detected,comparison):
    directory=OUT/'figures';baseline=baseline_rows(comparison)
    colors=dict(zip(('중앙','동측','서측','남측','북측'),('#386cb0','#f6a53a','#6ca448','#cf6689','#8b6bb2')))
    # Full image thumbnails only: no complete 3GB mosaic is read into memory.
    fig,ax=plt.subplots(figsize=(10,12))
    for o in ORTHO:
        signature=hashlib.sha256(json.dumps(stat_signature(o['path']),sort_keys=True).encode()).hexdigest()[:16]
        thumb=CACHE/'thumbnails'/(o['sheet']+'_'+signature+'.png');thumb.parent.mkdir(exist_ok=True)
        if thumb.exists():image=plt.imread(thumb)
        else:
            with rasterio.open(o['vrt']) as ds:
                ratio=max(ds.width,ds.height)/850
                arr=ds.read([1,2,3],out_shape=(3,int(ds.height/ratio),int(ds.width/ratio)),resampling=Resampling.nearest)
            image=arr.transpose(1,2,0);plt.imsave(thumb,image)
        left,bottom,right,top=o['geometry'].bounds
        ax.imshow(image,extent=(left,right,bottom,top),zorder=0)
        x,y=o['geometry'].exterior.xy;ax.plot(x,y,c='yellow',lw=.6);ax.annotate(o['sheet'],(left,top),fontsize=7,color='blue')
    for zone,g in PLOTS.groupby('zone'):ax.scatter(g.center_x,g.center_y,s=18,c=colors[zone],edgecolors='white',linewidths=.3,label=ZONE_EN[zone],zorder=4)
    ax.legend();ax.set_aspect('equal');ax.set_title('2021 orthophotos: nine manually referenced sheets / 50 field plots')
    ax.ticklabel_format(useOffset=False,style='plain');fig.savefig(directory/'overview_map.png',dpi=160,bbox_inches='tight');plt.close(fig)
    # Five zone representatives plus one additional opposite carbon extreme.
    low=baseline.sort_values('C_reference_tC_ha').iloc[0];high=baseline.sort_values('C_reference_tC_ha').iloc[-1]
    selected=[low.plot_id,high.plot_id]
    for zone,g in baseline.groupby('zone'):
        if zone not in set(baseline[baseline.plot_id.isin(selected)].zone):selected.append(g.sort_values('C_reference_tC_ha').iloc[len(g)//2].plot_id)
    for pid in baseline.sort_values('C_reference_tC_ha').plot_id:
        if len(selected)>=6:break
        if pid not in selected:selected.append(pid)
    assert len(selected)==6 and set(PLOTS[PLOTS.plot_id.isin(selected)].zone)==set(colors)
    pd.DataFrame({'plot_id':selected}).to_csv(OUT/'selected_overlay_plots.csv',index=False)
    for i,pid in enumerate(selected):
        p=PLOTS.set_index('plot_id').loc[pid];low_path=RASTERS[pid][0]
        fig,ax=plt.subplots(figsize=(8,8))
        with rasterio.open(low_path) as ds:show(ds,ax=ax)
        q=detected[detected.plot_id==pid]
        if len(q):q.boundary.plot(ax=ax,color='cyan',lw=.8)
        for geom,color,style in [(box(p.center_x-10,p.center_y-10,p.center_x+10,p.center_y+10),'yellow','-'),(p.geometry.buffer(30),'yellow','--')]:
            x,y=geom.exterior.xy;ax.plot(x,y,c=color,ls=style,lw=1.5)
        ax.scatter([p.center_x],[p.center_y],c='red',s=20,marker='+')
        ax.set_title(f'{ZONE_EN[p.zone]} {pid.split("-")[-1]} | cyan: crowns; yellow:20m square /30m circle')
        ax.ticklabel_format(useOffset=False,style='plain');fig.savefig(directory/f'{ZONE_EN[p.zone]}_{pid.split("-")[-1]}_overlay.png',dpi=160,bbox_inches='tight');plt.close(fig)
        if i<2:
            fig,ax=plt.subplots(figsize=(9,9))
            with rasterio.open(low_path) as ds:show(ds,ax=ax)
            extent=box(p.center_x-60,p.center_y-60,p.center_x+60,p.center_y+60)
            f=FOREST[FOREST.geometry.intersects(extent)];t=TRAILS[TRAILS.geometry.intersects(extent)]
            if len(f):f.boundary.plot(ax=ax,color='orange',lw=.8)
            if len(t):t.plot(ax=ax,color='cyan',lw=1.2)
            ax.set_xlim(p.center_x-60,p.center_x+60);ax.set_ylim(p.center_y-60,p.center_y+60)
            ax.set_title('Visual georeferencing check | orange:2025 forest map / cyan:trails')
            fig.text(.12,.02,'Reference bbox test is numerical; boundary alignment still requires visual review.',fontsize=9)
            fig.savefig(directory/f'georef_overlay_{i+1}.png',dpi=160,bbox_inches='tight');plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    for zone,g in baseline.groupby('zone'):
        for ax,column in zip(axes,('C_estimated_tC_ha','C_calibrated_OOF_tC_ha')):ax.scatter(g.C_reference_tC_ha,g[column],c=colors[zone],label=ZONE_EN[zone],s=24)
    for ax,column,title in zip(axes,('C_estimated_tC_ha','C_calibrated_OOF_tC_ha'),('Uncalibrated','Leave-one-zone-out calibrated')):
        limit=max(baseline.C_reference_tC_ha.max(),baseline[column].max())*1.06
        ax.plot([0,limit],[0,limit],'k--',lw=1);ax.set_xlim(0,limit);ax.set_ylim(0,limit);ax.set_aspect('equal')
        ax.set(xlabel='Field-derived aboveground carbon (tC/ha)',ylabel='Estimated aboveground carbon (tC/ha)',title=title);ax.legend(fontsize=8);ax.grid(alpha=.2)
    fig.tight_layout();fig.savefig(directory/'plot_carbon_scatter.png',dpi=170,bbox_inches='tight');plt.close(fig)
    g=baseline.groupby('zone')[['C_visible_tC_ha','C_occluded_tC_ha','n_visible','n_occluded','n_detected']].mean().reindex(colors)
    fig,axes=plt.subplots(1,2,figsize=(13,5));xx=np.arange(5)
    axes[0].bar(xx,g.C_visible_tC_ha,label='Visible by T1 minimum',color='#73af75');axes[0].bar(xx,g.C_occluded_tC_ha,bottom=g.C_visible_tC_ha,label='Occluded by T1 minimum',color='#d982a4')
    for j,(col,label,c) in enumerate([('n_visible','Field visible','#73af75'),('n_occluded','Field occluded','#d982a4'),('n_detected','Detected','#5a86b6')]):axes[1].bar(xx+(j-1)*.25,g[col],.24,label=label,color=c)
    for ax in axes:ax.set_xticks(xx,[ZONE_EN[z] for z in g.index]);ax.legend(fontsize=8);ax.grid(axis='y',alpha=.2)
    axes[0].set_ylabel('Mean field aboveground C (tC/ha)');axes[1].set_ylabel('Mean tree count per400m2')
    fig.tight_layout();fig.savefig(directory/'visible_occluded_bar.png',dpi=170,bbox_inches='tight');plt.close(fig)

def write_report(comparison,summary,detected,reference):
    base=summary[(summary.source=='금정산')&(summary.area_kind=='square20')&(summary.CF_scenario=='CF0.5')&(summary.height_scenario=='height0')&(summary.field_DBH_min_cm==10)]
    total=base[base.zone=='ALL'].iloc[0]
    proxy=detected.mapping.fillna('').str.contains('平均|평균',regex=True) if len(detected) else pd.Series([],dtype=bool)
    source_unknown=int(detected.get('forest_fallback',pd.Series([],dtype=bool)).fillna(False).astype(bool).sum())
    table=base[['zone','n','reference_mean','estimate_mean','estimate_reference_ratio','r','RMSE','MAE','calibrated_RMSE','calibrated_ratio']].to_markdown(index=False,floatfmt='.3f')
    sensitivity=summary[(summary.source=='금정산')&(summary.zone=='ALL')][['area_kind','CF_scenario','height_scenario','field_DBH_min_cm','reference_mean','estimate_mean','estimate_reference_ratio','RMSE','calibrated_RMSE']].to_markdown(index=False,floatfmt='.3f')
    report=f'''# 금정산 사진·높이 기반 지상부 탄소 비교

**50개20m 조사구, DBH≥10cm: 실측 기반 {total.reference_mean:.2f} / 영상 추정 {total.estimate_mean:.2f}tC/ha, 비율 {total.estimate_reference_ratio:.3f}, RMSE {total.RMSE:.2f}, zone OOF 보정 후 RMSE {total.calibrated_RMSE:.2f}tC/ha.**

## 전체·zone별 결과

{table}

모든 탄소값은 지상부(줄기+가지+잎)의 tC/ha이다. 뿌리·토양·고사목은 포함하지 않는다.
이전 금정산 A/B/C의 지상부+뿌리 탄소값과 계산 범위가 다르다.
논문 발표는 2011년이지만 실제 측정은 2009년 5월~2010년 9월이며, 정사영상은 2021년이다.

## NEON25cm 참고 기준선

같은 HF 커밋·score 0.2/patch 600·25cm→10cm 입력 실험의 추정/참조 비율은 {NEON_25CM_BASELINE['estimate_reference_ratio']:.6f},
F1_visible은 {NEON_25CM_BASELINE['F1_visible']:.6f}, 탄소 RMSE는 {NEON_25CM_BASELINE['carbon_RMSE_tC_ha']:.3f}tC/ha이다.
이 값은 보정계수로 사용하지 않았다. NEON은 지역·공식·참조 모집단이 달라 국내 성능을 보장하지 않는다.
금정산은 개별 줄기 좌표가 없어 precision/recall/F1이나 1:1 개체 매칭을 계산하지 않고, 탐지 수와 보이는 나무 수만 비교했다.

## 좌표와 영상

- 지정된 1:5000 도엽 규칙으로 경위도 모서리를 EPSG:5187로 변환했다. 좌상단은 (minx−50m,maxy+50m), 픽셀은 25cm이며 회전은 없다고 가정했다.
- 원본 TIFF 9개는 수정하지 않았다. 캐시 VRT에 좌표를 할당한 뒤 조사구 crop만 GeoTIFF로 저장했다.
- 35913003의 영상 전체를 EPSG:5179로 변환한 bbox가 기준의 각 변 3m 이내인지 assert했다. 9개 영상의 폭·높이 차이는 georef_check.csv에 기록했다.
- 각 중심 ±60m crop의 원본 25cm와 bilinear 10cm 확대본을 저장했다. 확대해도 선명도가 복원되지 않는다.
- Meta CHM은 대상 zoom9 quad만 사용하고 원격 COG의 필요한 window만 읽었다. 제품의 명목 1m 해상도와 native CRS 픽셀 간격을 구분하여 원본 tags를 보존했다.
  H는 native Meta 셀의 최대값으로 추출했다. 별도 EPSG:5187 1m CHM은 max 리샘플한 표시용이며 추론에는 쓰지 않았다.
- CHM 관측 메타데이터는 raster_manifest.csv와 meta_chm_sources.json에 있다. 날짜를 찾지 못하면 unknown으로 남기며, 파일 생성일과 촬영일을 구분했다.
- 임상도·등산로 겹침 그림 2개를 저장했다. 수치 bbox 검산은 현장 정합의 증명이 아니므로 사람이 그림을 확인해야 한다.

## 추정과 실측 공식

- HF모델:{HF_MODEL_PROVENANCE['model_name']} / 고정커밋:{HF_MODEL_PROVENANCE['resolved_revision']},DeepForest2.1.0.
- NEON의predict_crowns를재사용:score.2,patch600,overlap.25,NMS.15,상자중심영역판정,H≥3m,H=상자내native CHM최대,CD=상자가로·세로평균.
- Jucker:DBH(cm)=a×(H(m)×CD(m))^b×exp(g²/2).침엽biome21:{JUCKER[21]},활엽biome20:{JUCKER[20]}.
  혼효림은두DBH예측평균이다.수종미상은임상별대표(침엽:소나무류2식평균,활엽:낙엽활엽수평균,혼효:두그룹각0.5)를사용했다.
- 임상도밖/비산림·죽림등임상미확인탐지 {source_unknown}개는혼효대표대용으로계산하고forest_fallback으로표시했다.
  지도미피복을탄소0으로단정하거나탐지를숨겨삭제하지않았다.이대용의기여는mapping_contributions.csv.
- 국내2014상대생장식aD^b의줄기·가지·잎합을사용했다.그룹대용은계수평균이아닌각식의예측질량평균이다.
- 소나무D>40cm는강원식,나머지는중부식.떡갈/기타참나무는4종평균,개서어나무/소사나무는서어나무.
  야장의등나무와D<6cm는제외했다.필드기본은D≥10cm,보조는D≥6cm.범위밖은외삽+flag이다.
- 영상탐지는NEON과같이H≥3m만으로기본계산하며추정DBH<10cm도포함한다.추정DBH≥10cm만의탄소도별도열로저장해모집단차이를확인할수있다.
- CF기본.5,민감도는국내2010수종별목재CF를모든지상부부위에공통적용했다.미측정수종CF는.5로유지하고대용가중치를기록했다.
  대용그룹은각수종의AGB와그수종CF를먼저짝지어탄소를계산한뒤평균했다.CF열은그결과의유효CF이다.
- 교목층T1최소수고이상은위에서보이는나무로대리분류했다.이는실제수관가시성정답이아니며개체별겹침을알수없다.
- 원문표기의심개어서나무·대팻집·고뢰쇠나무는임의정정없이기타활엽수대용을적용했다.

## 보정과 민감도

zone 5개를 GroupKFold 5개로 나누어 매번 한 zone을 통째로 제외했다. 학습 4개 zone의 합계 실측 밀도/합계 추정 밀도로 보정계수를 구했다.
시험 zone의 정답은 계수에 사용하지 않았으며 train/test ID와 계수는 calibration_folds.csv에 보존했다.
학습 추정 합계가 0이면 계수를 구할 수 없어 NaN으로 표시한다. 이는 이 50개 자료 안의 공간 보정 시험이며 새 지역 성능 검증이 아니다.

{sensitivity}

30m 원은 추정 범위의 민감도다. 원 안의 나무를 새로 조사한 것이 아니므로 20m 조사구의 참조 밀도를 대용했다.
면적이 달라 탐지 수는 n_detected_per400m2도 함께 확인해야 한다. 높이 ±10%는 상자·탐지 H≥3m 필터·분석 영역을 고정하고 H→DBH→AGB만 변경했다.

## 해석상 한계

- 현장2009~2010과영상2021사이에11~12년성장·고사·교란이있다.논문연도2011과영상사이는약10년이다.CHM촬영일은별도메타데이터로확인해야한다.
- 초 단위 GPS의 약25~30m 오차는20m 조사구보다 크고, 조사구 회전 방향도 모른다. 축정렬20m 정사각형은 비교를 위한 근사이다. 원 GPS 측지계가 명시되지 않아 경위도는 WGS84(EPSG:4326)로 가정했다.
- 수치bbox1개일치는위치정확도인증이아니며,영상9개전체의회전·기하왜곡·50m외곽여유가완벽히같다는보장은없다.
- 실제국내25cm영상은인위적으로축소한NEON영상과센서·압축·촬영각·색감·계절이다르다.10cm확대는정보를복원하지않는다.
- MetaCHM자체모델오차·촬영시점·지형/수평정합오차가있고최대높이가이웃큰나무를반영할수있다.
- 임상도2025의임분수종을2021개체수종으로대용했다.혼효림구성비와개별수종은모르며경계/중복의영향이있다.
- Jucker구북구온대식과국내수종식이금정산/상자CD에맞는지는검증되지않았다.모든범위외삽과평균대용을표시했다.
- 비교대상의실측탄소역시상대생장식참조값으로실제무게측정값이아니다.수고에따른가시성대리분류와DBH10cm기준도불확실성을만든다.

## 검산과 출처

- plot_comparison.csv는기본50행,별도민감도long표는50개×5시나리오×2직경기준=500행이다.기본summary.csv는전체·5zone·NEON참고7행이다.
  zone별검증분리,400㎡/ha·kg/t·CF환산,수종평균식·DBH40cm분기검사를수행했다.
- [Jucker원문구현]({JUCKER_URL}),계수와SHA256은provenance.json.
- 국내2014/2010보고서의제공CSV와README를사용했다.나무별formula열에수종·부위·계수·가중치·원문쪽을기록했다.
- README의 가지 예시0.069×24^1.995는39.6kg가 아니라39.117448kg이다. 예시 산술 불일치를 기록하고 제공 CSV 계수는 그대로 사용했다.
- [Meta/WRI공개자료](https://registry.opendata.aws/dataforgood-fb-forests/),출처Maxar/Meta/WRI;관련관측메타데이터보존.
- 국토지리정보원정사영상XML2021,산림청임상도2025·등산로,서정범2011동아대박사부록야장(측정2009~2010).
- 계산·그림은Colab실행결과이며Windows검증은표준라이브러리와노트북구조확인에한정한다.학위논문자료는공개업로드하지않는다.
'''
    (OUT/'report.md').write_text(report,encoding='utf-8')
    pd.DataFrame(AUDIT,columns=['stage','item','message']).to_csv(OUT/'audit.csv',index=False)
    pd.DataFrame(NEON.AUDIT,columns=['stage','item','reason']).to_csv(OUT/'neon_inference_audit.csv',index=False)
    contributions=[]
    for label,df in [('detected_60m_crops',detected),('field_ge10',reference[reference.included_ge10])]:
        total=float(df.carbon_kg.sum())
        for rule,g in df.groupby('mapping'):
            contributions.append(dict(population=label,mapping=rule,n=len(g),carbon_kg=float(g.carbon_kg.sum()),carbon_share_pct=100*float(g.carbon_kg.sum())/total if total else np.nan,
                note='detected_60m includes per-plot buffers and possible repeated real trees; not a landscape total' if label.startswith('detected') else '50 original20m field plots'))
    pd.DataFrame(contributions).to_csv(OUT/'mapping_contributions.csv',index=False)
    json_save(OUT/'run_validation.json',dict(plots=50,reference_rows=len(reference),comparison_rows=len(comparison),georef_control_passed=True,
        no_root_biomass=True,calibration='leave-one-zone-out; train-only ratio',no_individual_F1=True,
        synthetic_and_unit_checks='passed before real inference',georef_visual_review='two figures saved; requires human review'))

def runtime_checks():
    dimensional_checks()
    test={20:dict(a=.694,b=.73,g=0),21:dict(a=1.004,b=.73,g=0)}
    broad,_=jucker_diameter(20,5,'2',test);con,_=jucker_diameter(20,5,'1',test);mixed,_=jucker_diameter(20,5,'3',test)
    assert np.isclose(mixed,(broad+con)/2)
    # Real Rasterio crop/grid test, tiny data only; no source image mutation.
    with tempfile.TemporaryDirectory() as folder:
        file=Path(folder)/'test.tif';arr=np.arange(12*12,dtype='uint8').reshape(12,12)
        tr=from_origin(200000,300000,.25,.25)
        with rasterio.open(file,'w',driver='GTiff',height=12,width=12,count=3,dtype='uint8',crs=CRS_WORK,transform=tr) as ds:ds.write(np.stack([arr]*3))
        with rasterio.open(file) as ds:
            up=ds.read(out_shape=(3,30,30),resampling=Resampling.bilinear)
            transformed=ds.transform*rasterio.Affine.scale(.4,.4)
            assert up.shape==(3,30,30) and np.allclose((transformed.a,abs(transformed.e)),.1)
            assert np.allclose(rasterio.transform.array_bounds(12,12,tr),rasterio.transform.array_bounds(30,30,transformed))
    assert ALLO.mapping('소나무',40)[0][0][0]=='중부지방소나무'
    assert ALLO.mapping('소나무',40.1)[0][0][0]=='강원지방소나무'
    assert np.isclose(ALLO.predict('떡갈나무',20)['AGB_kg'],np.mean([ALLO.predict(s,20)['AGB_kg'] for s in ALLO.oaks]))
    # Perturb held-out targets and rerun the actual calibration implementation.
    synthetic=pd.DataFrame(dict(plot_id=list('ABCDE'),zone=list('ABCDE'),C_reference_tC_ha=[10.,20,30,40,50],C_estimated_tC_ha=[5.,10,15,20,25]))
    before,_=calibrate_zone_holdout(synthetic)
    changed=synthetic.copy();changed.loc[changed.zone=='A','C_reference_tC_ha']=1e9
    after,_=calibrate_zone_holdout(changed)
    assert before.loc[0,'calibration_factor']==after.loc[0,'calibration_factor']==2
    assert before.loc[0,'C_calibrated_OOF_tC_ha']==after.loc[0,'C_calibrated_OOF_tC_ha']==10
    print('Colab 실제 라이브러리 격자/단위/수종/혼효림 예측량평균 검증 통과')

ZONE_EN={'중앙':'Central','동측':'East','서측':'West','남측':'South','북측':'North'}

## 05. 입력 확인·README·공식 원문 검증

필수파일과50개조사구를확인하고README를출력합니다.Jucker원문biome20/21을파싱하고계수·해시를기록합니다.지정숫자와다르면원문을사용하며경고합니다.

In [ ]:
initialize()
runtime_checks()
print('사용할 Jucker계수:',{k:JUCKER[k] for k in (20,21)})
display(PLOTS.drop(columns='geometry').head())

## 06. 도엽 좌표 할당·엄격한 범위 검증

원본TIFF를수정하지않고캐시VRT에5187좌표를할당합니다.35913003기준bbox각변3m이내와9개영상폭·높이차이를검사합니다.실패하면GPU추론전중단합니다.

In [ ]:
GEOREF_CHECK=georeference()
print('35913003 기준범위 대조:')
display(GEOREF_CHECK[GEOREF_CHECK.sheet=='35913003'])

## 07. 임상도·등산로 필드 확인

임상/수종필드의실제이름과예시를먼저출력합니다.코드필드만있으면명시한코드표를사용합니다.지도밖·비입목등은대용여부를표시합니다.

In [ ]:
read_vectors()
print('임상도/등산로:',len(FOREST),len(TRAILS))

## 08. 야장 참조 계산

기본≥10cm,보조≥6cm.줄기+가지+잎합×CF.5이며뿌리는제외합니다.T1최소수고이상/미만의개수와탄소를나눌수있도록보임표시를저장합니다.

In [ ]:
FIELD_REFERENCE=field_reference()
display(FIELD_REFERENCE[['plot_id','수종','DBH_cm','H_m','included_ge10','visible_by_T1','AGB_kg','carbon_kg','range_flag']].head())
print('원자료/6cm이상/10cm이상:',len(FIELD_REFERENCE),int(FIELD_REFERENCE.included_ge6.sum()),int(FIELD_REFERENCE.included_ge10.sum()))

## 09. 원본25cm·10cm확대본·CHM crop 캐시

각중심±60m,120m폭crop입니다.25cm사진은bilinear10cm로확대합니다.Meta CHMnative crop은H최대추출용,5187 1m CHM은별도표시용입니다.관측일을못찾으면unknown으로남깁니다.

In [ ]:
build_crops()
display(CHM_INFO[['plot_id','observation_dates','valid_fraction']])

## 10. 고정DeepForest 추론과국내공식

같은HF커밋,score.2/patch600,CHM≥3m.H·CD는NEON함수그대로,DBH는구북구20/21,AGB는국내2014지상부식으로교체합니다.원시탐지·처리결과를Drive에캐시합니다.

In [ ]:
DETECTED=estimate_boxes()
print('60m crop 안 탐지(비교영역필터전):',len(DETECTED))
display(DETECTED.drop(columns='geometry').head())

## 11. 조사구 비교·zone분리 보정·민감도

공식20m정사각형과30m원,CF,높이±10%,참조10/6cm를계산합니다.30m원의실측밀도는20m조사구대용입니다.보정계수는시험zone을뺀4개zone에서만적합합니다.NEON기준선은보정에쓰지않습니다.

In [ ]:
PLOT_COMPARISON,SUMMARY=compare_plots(DETECTED,FIELD_REFERENCE)
display(baseline_rows(PLOT_COMPARISON)[['plot_id','zone','C_reference_tC_ha','C_estimated_tC_ha','estimate_reference_ratio','C_calibrated_OOF_tC_ha']])

## 12. 지도·시각 정합·비교 그림

도엽9개와조사구50개overview,임상도·등산로겹침2장,zone별산점도,가시/가림막대,5zone를포함한저/고탄소6개overlay를저장합니다.실측개체줄기점은없으므로임의로나무위치를그리지않습니다.

In [ ]:
figures(DETECTED,PLOT_COMPARISON)
from IPython.display import display,Image
display(Image(filename=str(OUT/'figures/overview_map.png')))
for path in sorted((OUT/'figures').glob('georef_overlay_*.png')):
    display(Image(filename=str(path)))
print('임상도·등산로 정합은 위그림을사람이확인해야합니다.수치assert만으로정합을확정하지않습니다.')

## 13. 한국어 보고서·출처·검산 기록

맨위한줄요약에기본20m·≥10cm의결과를씁니다.좌표오차·시점·임상대용·원공식범위·NEON25cm차이를기록합니다.

In [ ]:
write_report(PLOT_COMPARISON,SUMMARY,DETECTED,FIELD_REFERENCE)
print('보고서:',OUT/'report.md')
print('생성파일:')
for p in sorted(OUT.rglob('*')):
    if p.is_file():print(p.relative_to(OUT))

## 14. 사람이 읽는 최종 요약

기본비교는400㎡·실측DBH≥10cm·CF.5·높이원값입니다.모든탄소와RMSE단위는tC/ha이며NEON기준선은참고행입니다.

In [ ]:
human=SUMMARY[((SUMMARY.source=='금정산')&(SUMMARY.area_kind=='square20')&(SUMMARY.CF_scenario=='CF0.5')&(SUMMARY.height_scenario=='height0')&(SUMMARY.field_DBH_min_cm==10))|(SUMMARY.source!='금정산')].copy()
human=human[['source','zone','n','reference_mean','estimate_mean','estimate_reference_ratio','r','RMSE','calibrated_RMSE','F1_visible']]
human=human.rename(columns={'source':'자료','zone':'구역','n':'조사구수','reference_mean':'실측기반평균(tC/ha)','estimate_mean':'추정평균(tC/ha)','estimate_reference_ratio':'추정/실측','r':'상관r','RMSE':'RMSE(tC/ha)','calibrated_RMSE':'보정후RMSE(tC/ha)','F1_visible':'NEON_F1참고'})
display(human.round(3))
print('NEON25cm 참고비율:',NEON_25CM_BASELINE['estimate_reference_ratio'],'(보정 미사용)')

## 15. 결과ZIP·Drive사본·다운로드

결과폴더만ZIP으로묶습니다.수GB원본정사영상과crop/탐지캐시는Drive에보존하고ZIP에중복포함하지않습니다.

In [ ]:
from google.colab import files
archive=shutil.make_archive('/content/geumjeongsan_results','zip',root_dir=str(OUT))
shutil.copy2(archive,ROOT/'outputs/geumjeongsan_results.zip')
print('ZIP:',archive,'Drive 사본:',ROOT/'outputs/geumjeongsan_results.zip')
files.download('/content/geumjeongsan_results.zip')